# Phase 4 -- Notebook 1: D, C and the distilled student (S1, S2, S5)

Independent of the other notebook: downloads the Phase 4 data itself and needs no other notebook's output.

| Run | Phase-1 model | Method | Submit? |
|---|---|---|---|
| **S1** | D encoder (~100M) | **LoRMA** (rank 16, all attention + MLP layers) | yes |
| **S2** | C (113M) | **BitFit** (only the bias terms train; C was built with biases for this) | yes |
| **S5** | C cut to its first 6 layers (~62M) | **Knowledge distillation** from S1 + S2 (~213M): teacher probabilities + where the teachers look across the chunks | yes |
| R1 | D encoder | Full fine-tuning -- reference: does LoRMA keep full fine-tuning's score? | no (dev only) |
| R2 | same student as S5 | Gold labels only -- control: what does distillation add? | no (dev only) |

**Pipeline (every run)** -- cases are long, so each is cut into 512-token chunks (64-token overlap; the first
chunk + up to 15 from the end).
1. **Adapt** the Phase 1 model with its method on one chunk per case per step (the last chunk half the time,
   otherwise a random one), labelled with the case outcome, on 80% of train.
2. **Encode** every chunk of the held-out 20% of train, dev and test with the adapted model.
3. **Read the whole case**: 7 small readers over the chunk vectors + chunk positions (last chunk, first chunk,
   mean, max, attention, GRU + attention, transformer) and a section study (first / middle / last third, last 4
   chunks), trained on the held-out 20% only and chosen on a validation slice of it -- dev stays clean.
4. **Calibrate**: decision threshold fitted on dev, used only if a 2-fold dev check agrees.

**How to run**
1. Inputs: `legallm-tokenizers` (for `t2_unigram.model`) and the datasets holding the **C** and **D** checkpoints (file names keep their `A_step... / B_step... / C_step... / D_step...` start;
   `.pt`, `.zip` or a folder Kaggle unzipped are all fine).
2. Secret on: `HF`. Accelerator **GPU T4 x2**, **internet on**.
3. **Save Version -> Save & Run All.** Time: ~4.5-5 h (S1 | S2, then R1 | S5, then R2; every run has a time budget).
4. Submit `p4_nb1/S1/submission.zip`, `p4_nb1/S2/submission.zip`, `p4_nb1/S5/submission.zip`. Download `p4_nb1/ens_nb1.json` too
   (dev/test scores of every run, for the ensemble submission S6).


## 1. Inputs: tokenizer and checkpoints

In [ ]:
import csv
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

import numpy as np
import torch

N_GPUS = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "p4_nb1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / "ckpts"
CKPT_DIR.mkdir(exist_ok=True)


def step_of(p):
    return int(Path(p).name.split("_step")[1].split("_")[0].split(".")[0])


def _rebuild_from_folder(folder, dest):
    """Kaggle unzips an uploaded checkpoint (a .pt file is a zip archive inside),
    leaving a folder with data.pkl, data/0, version, ... Re-pack it into a .pt."""
    pkl = next(Path(folder).rglob("data.pkl"))
    root = pkl.parent
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(root.rglob("*")):
            if f.is_file():
                zf.write(f, arcname="archive/" + f.relative_to(root).as_posix())
    return dest


def _collect(prefix):
    """{name: path} of usable <prefix>_step*.pt checkpoints under /kaggle/input:
    a .pt file; a .zip holding the .pt; the .pt renamed .zip; or a folder that
    Kaggle unzipped from an uploaded checkpoint (re-packed here)."""
    found = {}
    if not KAGGLE_INPUT.exists():
        return found
    for p in KAGGLE_INPUT.rglob(f"{prefix}_step*"):
        stem = p.name[:-3] if p.name.endswith(".pt") else (p.name[:-4] if p.name.endswith(".zip") else p.name)
        dest = CKPT_DIR / (stem + ".pt")
        if p.is_file() and p.suffix == ".pt":
            found.setdefault(p.name, p)
        elif p.is_dir() and any(p.rglob("data.pkl")) and dest.name not in found:
            found[dest.name] = _rebuild_from_folder(p, dest)
            print(f"  re-packed unzipped checkpoint folder {p} -> {dest.name}")
    for z in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                inner = [n for n in names if Path(n).name.startswith(f"{prefix}_step") and n.endswith(".pt")]
                for n in inner:
                    dest = CKPT_DIR / Path(n).name
                    if dest.name not in found:
                        with zf.open(n) as src, open(dest, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        found[dest.name] = dest
                if not inner and z.name.startswith(f"{prefix}_step") and any(n.endswith("data.pkl") for n in names):
                    dest = CKPT_DIR / (z.stem + ".pt")
                    if dest.name not in found:
                        shutil.copy(z, dest)
                        found[dest.name] = dest
        except zipfile.BadZipFile:
            continue
    return found




def checkpoints(prefix):
    found = _collect(prefix)
    assert found, f"No {prefix}_step* checkpoint (.pt, .zip or unzipped folder) under /kaggle/input -- attach it."
    return max(found.values(), key=step_of)


def find_one(name, label):
    hits = sorted(KAGGLE_INPUT.rglob(name)) if KAGGLE_INPUT.exists() else []
    assert hits, f"{name} not found under /kaggle/input -- attach {label}."
    return hits[0]


TOK_PATH = find_one("t2_unigram.model", "legallm-tokenizers")
CKPT = {"C": str(checkpoints("C")), "D": str(checkpoints("D"))}
for k, p in CKPT.items():
    print(f"{k}: {Path(p).name}")


## 2. Code files (written to /kaggle/working)

In [ ]:
%%writefile /kaggle/working/legal_d.py
"""legal_d.py -- Model D: encoder-decoder legal language model + its trainer.

Model D (T5 / RedLLM-style encoder-decoder, tokenizer T2):
  * Encoder: bidirectional self-attention over the prompt.
  * Decoder: causal self-attention over the continuation + cross-attention
    to the encoder output.
  * Shared (tied) embedding for encoder input, decoder input and output head.
  * RMSNorm (pre-norm), RoPE in self-attention, QK-norm, GQA, SwiGLU, no biases.

Trainer: one process per GPU (DDP via torch.distributed.run), Muon optimizer
for the hidden weight matrices + AdamW for embeddings/norms, fp16 autocast
with GradScaler (T4 has no bf16), time-based warmup-stable-decay LR
schedule (so the LR always finishes its decay when the time budget ends),
flat checkpoints with resume, periodic push to a Kaggle dataset.

Run training:   python -m torch.distributed.run --standalone --nproc_per_node=2 legal_d.py d_config.json
Import helpers: import legal_d  (model, generation, scoring)
"""
import contextlib
import datetime
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

ARCH_ID = "D"

MODEL_D_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram), same as C
    d_model=768,
    n_enc_layer=12,
    n_dec_layer=12,
    n_head=12,
    n_kv_head=4,              # GQA 3:1 (self- and cross-attention)
    d_head=64,
    d_ff=2048,                # SwiGLU (same FFN parameter count as C's 4x GELU)
    max_enc_len=1024,
    max_dec_len=1024,
    qk_norm=True,
    mlp="swiglu",
    rope_theta=10_000.0,
    tokenizer_id="T2",
    arch="encoder-decoder",
    tied_embeddings=True,
)

# ---------------------------------------------------------------------------
# Tokenizer (T2)
# ---------------------------------------------------------------------------
sp = None
EOS_ID = None
DEC_START_ID = None
BANNED_IN_OUTPUT = []


def load_tokenizer(path):
    global sp, EOS_ID, DEC_START_ID, BANNED_IN_OUTPUT
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(str(path))
    EOS_ID = sp.eos_id()
    assert EOS_ID >= 0, "T2 tokenizer has no <eos>"
    DEC_START_ID = sp.bos_id() if sp.bos_id() >= 0 else EOS_ID      # decoder start token
    BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
    return sp


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)


def rope_cache(n, d_head, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(n).float(), inv_freq)
    return freqs.cos(), freqs.sin()                      # [n, d_head/2]


def apply_rope(x, cos, sin):
    """x [B, H, T, d]; cos/sin [T, d/2]. Rotate-half form, computed in fp32."""
    cos, sin = cos[None, None].float(), sin[None, None].float()
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half].float(), x[..., half:].float()
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def expand_kv(t, n_rep):
    if n_rep == 1:
        return t
    B, H, T, D = t.shape
    return t[:, :, None].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class KVCache:
    """Decoder self-attention cache (allocated on first use)."""

    def __init__(self, max_len):
        self.max_len, self.pos, self.k, self.v = max_len, 0, None, None

    def update(self, k, v):
        if self.k is None:
            B, H, _, D = k.shape
            self.k = torch.zeros(B, H, self.max_len, D, device=k.device, dtype=k.dtype)
            self.v = torch.zeros_like(self.k)
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.causal, self.n_head, self.n_kv_head, self.d_head = causal, h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, key_mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        first_chunk = cache is None or cache.pos == 0
        if cache is not None:
            assert T == 1 or cache.pos == 0, "cached decoding feeds one token at a time"
            k, v = cache.update(k, v)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(k, rep), expand_kv(v, rep)
        if key_mask is not None:                                  # [B, Tk] True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1 and first_chunk)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output (no RoPE: T5-style)."""

    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.k_norm is not None:
            k = self.k_norm(k)
        return k, v

    def forward(self, x, enc_kv, enc_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(enc_kv[0], rep), expand_kv(enc_kv[1], rep)
        mask = enc_mask[:, None, None, :] if enc_mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg["d_model"], cfg["d_ff"]
        self.gate_proj = nn.Linear(d, f, bias=False)
        self.up_proj = nn.Linear(d, f, bias=False)
        self.down_proj = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = SelfAttention(cfg, causal=False)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, key_mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, key_mask)
        return x + self.mlp(self.mlp_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg["d_model"])
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg["d_model"])
        self.cross_attn = CrossAttention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, enc_out=None, enc_mask=None, enc_kv=None, cache=None):
        if enc_kv is None:
            enc_kv = self.cross_attn.project_kv(enc_out)
        x = x + self.self_attn(self.self_norm(x), cos, sin, cache=cache)
        x = x + self.cross_attn(self.cross_norm(x), enc_kv, enc_mask)
        return x + self.mlp(self.mlp_norm(x))


def _ce_chunk(h, w, t):
    logits = F.linear(h, w).float()
    lse = torch.logsumexp(logits, dim=-1)
    valid = (t != -100).float()
    nll = (lse - logits.gather(-1, t.clamp(min=0)[:, None]).squeeze(-1)) * valid
    return torch.stack([nll.sum(), (lse.pow(2) * valid).sum()])


def chunked_loss(h, w, targets, chunk=4096):
    """[1, 3] = [sum of token NLL, number of scored tokens, sum of logsumexp^2].
    Logits are built chunk by chunk (and recomputed in backward), so the full
    [tokens x 32k] logit tensor is never stored."""
    h = h.reshape(-1, h.shape[-1])
    t = targets.reshape(-1)
    parts = []
    for s in range(0, h.shape[0], chunk):
        hs, ts = h[s:s + chunk], t[s:s + chunk]
        if torch.is_grad_enabled() and h.requires_grad:
            parts.append(grad_checkpoint(_ce_chunk, hs, w, ts, use_reentrant=False))
        else:
            parts.append(_ce_chunk(hs, w, ts))
    tot = torch.stack(parts).sum(0)
    count = (t != -100).sum().to(tot.dtype)
    return torch.stack([tot[0], count, tot[1]]).unsqueeze(0)


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, V = config["d_model"], config["vocab_size"]
        self.embed = nn.Embedding(V, d)
        self.enc_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["n_enc_layer"])])
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config["n_dec_layer"])])
        self.dec_norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.embed.weight                  # tied
        cos, sin = rope_cache(max(config["max_enc_len"], config["max_dec_len"]), config["d_head"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_ckpt = False
        self.amp = True                                        # fp16 autocast on GPU (fp32 weights)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        for name, p in self.named_parameters():                 # scaled residual-output init
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                n_res = 2 * config["n_enc_layer"] if name.startswith("enc_") else 3 * config["n_dec_layer"]
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    def _run(self, blk, *args, **kwargs):
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            return grad_checkpoint(blk, *args, use_reentrant=False, **kwargs)
        return blk(*args, **kwargs)

    def encode(self, input_ids, attention_mask=None):
        """Contract method: encoder hidden states [B, T, d]. attention_mask
        (1 = real token) marks padding; positions start at 0 in every row
        (pad on the right)."""
        T = input_ids.shape[1]
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = attention_mask.bool() if attention_mask is not None else None
        for blk in self.enc_blocks:
            x = self._run(blk, x, cos, sin, key_mask)
        return self.enc_norm(x)

    def decode_hidden(self, dec_ids, enc_out, enc_mask=None):
        T = dec_ids.shape[1]
        x = self.embed(dec_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        m = enc_mask.bool() if enc_mask is not None else None
        for blk in self.dec_blocks:
            x = self._run(blk, x, cos, sin, enc_out=enc_out, enc_mask=m)
        return self.dec_norm(x)

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        """No targets -> logits [B, Q, V]. With targets -> [1, 3] sums
        (NLL, count, lse^2); targets = -100 are ignored."""
        use_amp = enc_ids.is_cuda and self.embed.weight.dtype == torch.float32 and self.amp
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            enc_out = self.encode(enc_ids, enc_mask)
            h = self.decode_hidden(dec_ids, enc_out, enc_mask)
            if targets is None:
                return self.head(h)
            return chunked_loss(h, self.head.weight, targets)

    def decoder_step(self, tok, pos, caches, cross_kvs, enc_mask):
        x = self.embed(tok)
        cos, sin = self.rope_cos[pos:pos + tok.shape[1]], self.rope_sin[pos:pos + tok.shape[1]]
        for blk, cache, kv in zip(self.dec_blocks, caches, cross_kvs):
            x = blk(x, cos, sin, enc_mask=enc_mask, enc_kv=kv, cache=cache)
        return self.head(self.dec_norm(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (one prompt, greedy): returns prompt + continuation ids."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


def build_model(config):
    """Contract function."""
    return EncoderDecoderLM(config)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample_top_p(logits, temperature, top_p, generator):
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_p < 1.0:
        sorted_p, idx = probs.sort(dim=-1, descending=True)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, idx, sorted_p)
    return torch.multinomial(probs, 1, generator=generator).squeeze(-1)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_enc_len=768, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None, sample=False,
                   temperature=1.0, top_p=1.0, generator=None, start_id=None):
    """Continuations for several prompts at once. Encoder input is right-padded
    (masked); every decoder row starts at the same position, so no decoder
    padding is needed. Greedy by default; `sample=True` gives temperature /
    top-p sampling (used for MBR candidates). Optional repetition penalty over
    generated tokens, n-gram blocking, and early stop once every row has more
    than `target_words` words."""
    model.eval()
    device = model.embed.weight.device
    cfg = model.config
    start_id = DEC_START_ID if start_id is None else start_id
    max_enc_len = min(max_enc_len or cfg["max_enc_len"], cfg["max_enc_len"])
    prompts = [(p[-max_enc_len:] if p else [EOS_ID if EOS_ID is not None else 0]) for p in prompt_id_lists]
    B, P = len(prompts), max(len(p) for p in prompts)
    enc_ids = torch.zeros(B, P, dtype=torch.long, device=device)
    enc_mask = torch.zeros(B, P, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        enc_ids[b, :len(p)] = torch.tensor(p, device=device)
        enc_mask[b, :len(p)] = True
    use_amp = device.type == "cuda" and model.embed.weight.dtype == torch.float32 and getattr(model, "amp", True)
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
    with amp:
        enc_out = model.encode(enc_ids, enc_mask)
        cross_kvs = [blk.cross_attn.project_kv(enc_out) for blk in model.dec_blocks]
        caches = [KVCache(max_new_tokens + 1) for _ in model.dec_blocks]
        tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
        logits = model.decoder_step(tok, 0, caches, cross_kvs, enc_mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = _sample_top_p(logits, temperature, top_p, generator) if sample else logits.argmax(dim=-1)
        for b, tk in enumerate(nxt.tolist()):
            generated[b].append(tk)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(len(decode(g).split()) > target_words for g in generated):
            break
        with amp:
            logits = model.decoder_step(nxt[:, None], t + 1, caches, cross_kvs, enc_mask)[:, -1, :].float()
    return generated


# ---------------------------------------------------------------------------
# Scoring helpers (mirror evaluate_phase1.py; BERTScore uses roberta-large)
# ---------------------------------------------------------------------------
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    n = lcs_length(pt, rt)
    if not n:
        return 0.0
    pr, rc = n / len(pt), n / len(rt)
    return 2 * pr * rc / (pr + rc)


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = _ngrams(p, n), _ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)


def score_predictions(pred, truth, ids):
    try:
        import bert_score
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True)
        import bert_score
    hyps = [pred[i]["text"].strip() for i in ids]
    refs = [truth[i]["text"].strip() for i in ids]
    _, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {"bertscore": float(f1.mean()),
            "rouge_l": sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs),
            "bleu4": bleu4([h.split() for h in hyps], [r.split() for r in refs])}


def to_words(toks, n_words=128):
    words = decode(toks).split()[:n_words]
    return " ".join(words) if words else "."


def predict(model, prompt_dict, ids, batch_size=16, desc="predict", **settings):
    """Greedy/sampled continuations -> {id: {"text": 128 words}}. Prompts are
    sorted by length so each batch has similar lengths."""
    from tqdm.auto import tqdm
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_size), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_size]
        for i, toks in zip(chunk, generate_batch(model, [enc[i] for i in chunk], **settings)):
            out[i] = {"text": to_words(toks)}
    return {i: out[i] for i in ids}


def mbr_pick(candidates):
    """Minimum-Bayes-risk choice: the candidate with the highest average
    ROUGE-L F1 against the other candidates (a consensus pick; no outside
    model involved)."""
    if len(candidates) <= 2:
        return candidates[0]
    best, best_u = 0, -1.0
    for i, c in enumerate(candidates):
        u = sum(rouge_l_f1(c, o) for j, o in enumerate(candidates) if j != i) / (len(candidates) - 1)
        if u > best_u:
            best, best_u = i, u
    return candidates[best]


def predict_mbr(model, prompt_dict, ids, n_samples=7, batch_prompts=8, greedy_settings=None,
                sample_settings=None, seed=1234, desc="mbr"):
    """For each prompt: 1 greedy candidate + n_samples sampled candidates ->
    MBR pick. Candidates of `batch_prompts` prompts are generated together."""
    from tqdm.auto import tqdm
    device = model.embed.weight.device
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_prompts), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_prompts]
        greedy = generate_batch(model, [enc[i] for i in chunk], **greedy_settings)
        sampled = generate_batch(model, [enc[i] for i in chunk for _ in range(n_samples)], sample=True,
                                 generator=gen, **sample_settings)
        for k, i in enumerate(chunk):
            cands = [to_words(greedy[k])] + [to_words(t) for t in sampled[k * n_samples:(k + 1) * n_samples]]
            out[i] = {"text": mbr_pick(cands)}
    return {i: out[i] for i in ids}


@torch.no_grad()
def dev_loss_bpb(model, dev_prompts, dev_truth, ids, max_enc_len=768, max_dec_len=512):
    """Continuation loss per token and bits-per-byte (tokenizer-independent)."""
    model.eval()
    device = model.embed.weight.device
    nll, n_tok, n_bytes = 0.0, 0, 0
    for i in ids:
        cont_text = " " + dev_truth[i]["text"].strip()
        cont = encode(cont_text)[:max_dec_len]
        ctx = encode(dev_prompts[i]["text"])[-max_enc_len:]
        if not cont or not ctx:
            continue
        enc_ids = torch.tensor([ctx], device=device)
        dec_in = torch.tensor([[DEC_START_ID] + cont[:-1]], device=device)
        tgt = torch.tensor([cont], device=device)
        out = model(enc_ids, dec_in, targets=tgt)[0]
        nll += out[0].item()
        n_tok += len(cont)
        n_bytes += len(cont_text.encode("utf-8"))
    return nll / max(1, n_tok), nll / (math.log(2) * max(1, n_bytes))


def self_test(device):
    """Checks: cached decoding == full forward; decoder is causal; encoder is
    two-way; encoder padding changes nothing; loss sums are correct; batched
    generation == one-at-a-time; Muon orthogonalises."""
    torch.manual_seed(0)
    tiny = dict(MODEL_D_CONFIG, vocab_size=101, d_model=64, n_enc_layer=2, n_dec_layer=2, n_head=4,
                n_kv_head=2, d_head=16, d_ff=96, max_enc_len=64, max_dec_len=64)
    m = build_model(tiny).to(device).eval()
    m.amp = False                                          # exactness checks in fp32 (GPU kernels still used)
    enc = torch.randint(4, 101, (2, 20), device=device)
    dec = torch.randint(4, 101, (2, 12), device=device)
    with torch.no_grad():
        full = m(enc, dec).float()
        enc_out = m.encode(enc)
        kvs = [blk.cross_attn.project_kv(enc_out) for blk in m.dec_blocks]
        caches = [KVCache(12) for _ in m.dec_blocks]
        steps = [m.decoder_step(dec[:, t:t + 1], t, caches, kvs, None) for t in range(12)]
        diff = (full - torch.cat(steps, 1).float()).abs().max().item()
        dec2 = dec.clone()
        dec2[:, 8] = (dec[:, 8] + 1) % 97 + 4
        leak = (m(enc, dec2)[:, :8] - full[:, :8]).abs().max().item()
        enc2 = enc.clone()
        enc2[:, 15] = (enc[:, 15] + 1) % 97 + 4
        two_way = (m.encode(enc2)[:, 2] - enc_out[:, 2]).abs().max().item()
        padded = torch.cat([enc, torch.zeros(2, 5, dtype=torch.long, device=device)], 1)
        mask = torch.cat([torch.ones(2, 20), torch.zeros(2, 5)], 1).to(device)
        pad_diff = (m(padded, dec, enc_mask=mask) - full).abs().max().item()
        tgt = torch.randint(4, 101, (2, 12), device=device)
        tgt[0, 3] = -100
        s = m(enc, dec, targets=tgt)[0]
        ref = F.cross_entropy(full.reshape(-1, 101), tgt.reshape(-1), ignore_index=-100, reduction="sum")
    assert diff < 1e-3, f"cached decoding != full forward ({diff})"
    assert leak < 1e-5, f"decoder sees future tokens ({leak})"
    assert two_way > 1e-5, "encoder is not bidirectional"
    assert pad_diff < 1e-4, f"encoder padding changes the output ({pad_diff})"
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 23, "loss sums wrong"
    pa, pb = enc[0].tolist(), enc[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1)
    alone = [generate_batch(m, [pa], max_new_tokens=8, start_id=1)[0],
             generate_batch(m, [pb], max_new_tokens=8, start_id=1)[0]]
    assert both == alone, "batched generation differs from one-at-a-time"
    g = torch.Generator(device=device)
    g.manual_seed(0)
    smp = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1, sample=True, temperature=0.8, top_p=0.9,
                         generator=g)
    assert len(smp) == 2 and all(len(x) == 8 for x in smp)
    X = newton_schulz5(torch.randn(48, 96, device=device)).float()
    sv = torch.linalg.svdvals(X)
    assert 0.3 < sv.min().item() and sv.max().item() < 1.5, f"Muon orthogonalisation off: {sv.min():.2f}..{sv.max():.2f}"
    print(f"Self-test passed: cached==full {diff:.1e}, causal decoder {leak:.1e}, two-way encoder, "
          f"padding-safe {pad_diff:.1e}, loss sums ok, batched==single, sampling ok, "
          f"Muon singular values {sv.min():.2f}..{sv.max():.2f}.")


# ---------------------------------------------------------------------------
# Muon optimizer (Jordan et al. 2024; update scale from Liu et al. 2025,
# "Muon is Scalable for LLM Training": 0.2 * sqrt(max(rows, cols)) so the
# learning rate and weight decay work on AdamW's scale).
# ---------------------------------------------------------------------------
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + eps)                              # normalise in fp32 (tiny grads survive)
    if G.is_cuda:
        X = X.half()                                       # T4: fast fp16 matmuls, values stay <= 1
    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        Bm = b * A + c * (A @ A)
        X = a * X + Bm @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.95, nesterov=True, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad.float()
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom) if group["nesterov"] else buf
                u = newton_schulz5(g, steps=group["ns_steps"]).float()
                p.mul_(1.0 - lr * wd)
                p.add_(u, alpha=-lr * 0.2 * math.sqrt(max(p.shape[0], p.shape[1])))
        return None


def build_optimizers(model, kind, lr, weight_decay):
    muon_params, adam_decay, adam_no_decay = [], [], []
    for name, p in model.named_parameters():
        if p.dim() == 2 and (name.startswith("enc_blocks") or name.startswith("dec_blocks")) and kind == "muon":
            muon_params.append(p)
        elif p.dim() >= 2:
            adam_decay.append(p)                           # embedding (tied output head); all matrices for AdamW
        else:
            adam_no_decay.append(p)                        # norm weights
    adam = torch.optim.AdamW([{"params": adam_decay, "weight_decay": weight_decay},
                              {"params": adam_no_decay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95), eps=1e-8)
    return ([Muon(muon_params, lr=lr, weight_decay=weight_decay)] if muon_params else []) + [adam]


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def lr_factor(step, progress, warmup_steps, cooldown_frac, floor):
    """Warmup-stable-decay on *time*: linear warmup (steps), constant, then a
    1-sqrt cooldown over the last `cooldown_frac` of the training time."""
    w = min(1.0, (step + 1) / max(1, warmup_steps))
    start = 1.0 - cooldown_frac
    if progress < start:
        return w
    r = min(1.0, (progress - start) / max(1e-9, cooldown_frac))
    return w * (floor + (1.0 - floor) * (1.0 - math.sqrt(r)))


def loss_from(out, z_loss_coef):
    s = out.sum(0)
    n = s[1].clamp(min=1)
    return s[0] / n, (s[0] + z_loss_coef * s[2]) / n


# ---------------------------------------------------------------------------
# Data: stream of 1024-token windows (round-robin over the cleaned files,
# documents separated by <eos>), byte positions saved for resume. Each GPU
# (rank) starts at a different offset of every file.
# ---------------------------------------------------------------------------
class WindowStream:
    def __init__(self, file_paths, window, rank=0, world=1, start_positions=None):
        self.files = [Path(p) for p in file_paths if Path(p).stat().st_size > 0]
        self.window = window
        self.positions, self.unaligned = {}, set()
        for p in self.files:
            if start_positions and p.name in start_positions:
                self.positions[p.name] = int(start_positions[p.name])
            else:
                self.positions[p.name] = (p.stat().st_size * rank) // world
                if self.positions[p.name] > 0:
                    self.unaligned.add(p.name)

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.files]
        for reader in itertools.cycle(readers):
            ids = encode(next(reader))
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.window:
                yield buffer[:self.window]
                buffer = buffer[self.window:]

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                if name in self.unaligned:                  # started mid-line: skip to the next line
                    pos += len(f.readline())
                    self.unaligned.discard(name)
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0


def make_example_batch(windows, P, Q, device):
    """windows: list of 1024-token lists -> encoder ids [B,P], decoder input
    [B,Q] (start token + shifted continuation), targets [B,Q]."""
    w = torch.tensor(windows, dtype=torch.long)
    enc = w[:, :P]
    tgt = w[:, P:P + Q]
    dec_in = torch.cat([torch.full((w.shape[0], 1), DEC_START_ID, dtype=torch.long), tgt[:, :-1]], dim=1)
    return enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), tgt.to(device, non_blocking=True)


def choose_shape(shapes, seed, step, micro):
    """Same (prompt, continuation) split on every GPU for a given micro-step."""
    rnd = random.Random(seed * 1_000_003 + step * 97 + micro)
    return rnd.choices([(s[0], s[1]) for s in shapes], weights=[s[2] for s in shapes])[0]


# ---------------------------------------------------------------------------
# Checkpoints and Kaggle push
# ---------------------------------------------------------------------------
def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix, search_dirs, extract_to):
    cands = []
    for d in search_dirs:
        d = Path(d)
        if d.exists():
            cands += [p for p in d.rglob(f"{prefix}_step*.pt")]
    best_zip = None
    for d in search_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for zpath in d.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath) as zf:
                    for name in zf.namelist():
                        base = Path(name).name
                        if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                            if best_zip is None or step_of(base) > best_zip[0]:
                                best_zip = (step_of(base), zpath, name)
            except zipfile.BadZipFile:
                continue
    best_file = max(cands, key=step_of) if cands else None
    if best_zip is not None and (best_file is None or best_zip[0] > step_of(best_file)):
        _, zpath, name = best_zip
        dest = Path(extract_to) / Path(name).name
        with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return dest
    return best_file


def prune_old_checkpoints(ckpt_dir, prefix, keep=2):
    ckpts = sorted(Path(ckpt_dir).glob(f"{prefix}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(output_dir, dataset_id, message):
    kdir = Path.home() / ".kaggle"
    if not ((kdir / "access_token").exists() or (kdir / "kaggle.json").exists() or os.environ.get("KAGGLE_API_TOKEN")):
        print("Skipping checkpoint push: no Kaggle API token.", flush=True)
        return False
    slug = dataset_id.split("/")[1]
    with open(Path(output_dir) / "dataset-metadata.json", "w") as f:
        json.dump({"title": slug, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    try:
        exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
                                timeout=300).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", str(output_dir), "-m", message, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", str(output_dir), "-r", "zip"])
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    except (OSError, subprocess.TimeoutExpired) as exc:
        print(f"Checkpoint push failed: {exc}", flush=True)
        return False
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}", flush=True)
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).", flush=True)
    return True


def load_for_inference(path, device, half=False):
    """fp32 weights + fp16 autocast on GPU (same numerics as training)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(ckpt["config"])
    model.load_state_dict(ckpt["model_state"])
    meta = {k: ckpt.get(k) for k in ("step", "tokens_seen", "config", "run_id")}
    del ckpt
    model = model.to(device).eval()
    if half and device.type == "cuda":
        model = model.half()
        model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
    return model, meta


# ---------------------------------------------------------------------------
# Trainer (one process per GPU)
# ---------------------------------------------------------------------------
def _probe_micro_batch(model, cfg, device, P, Q):
    """Largest micro-batch (1024-token examples) that fits the memory budget
    on this GPU, with the real optimizers; weights are restored afterwards."""
    saved = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
    results = {}
    for bs in cfg["candidate_micro_bs"]:
        optims = build_optimizers(model, cfg["optimizer"], 1e-5, 0.0)
        scaler = make_grad_scaler(True)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        try:
            windows = [[random.randrange(4, 1000) for _ in range(P + Q)] for _ in range(bs)]
            enc, dec_in, tgt = make_example_batch(windows, P, Q, device)
            times = []
            for _ in range(3):
                torch.cuda.synchronize(device)
                t0 = time.time()
                for o in optims:
                    o.zero_grad(set_to_none=True)
                _, total = loss_from(model(enc, dec_in, targets=tgt), 1e-4)
                scaler.scale(total).backward()
                for o in optims:
                    scaler.step(o)
                scaler.update()
                torch.cuda.synchronize(device)
                times.append(time.time() - t0)
            peak = torch.cuda.max_memory_allocated(device) / 1e9
            ok = peak <= cfg["memory_budget_gb"]
            results[bs] = (peak, min(times[1:]), ok)
            if not ok:
                break
        except Exception as exc:                           # noqa: BLE001
            if "out of memory" not in str(exc).lower():
                raise
            results[bs] = (None, None, False)
            break
        finally:
            for o in optims:
                o.zero_grad(set_to_none=True)
            del optims
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    model.load_state_dict(saved)
    return results


def train_main(config_path):
    cfg = json.load(open(config_path))
    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=90))
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", 0))
    else:
        rank, world, local_rank = 0, 1, 0
    device = torch.device(f"cuda:{local_rank}") if cuda else torch.device("cpu")
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0

    def log(*a):
        if main:
            print(*a, flush=True)

    load_tokenizer(cfg["tokenizer_path"])
    model_cfg = cfg["model"]
    prefix = cfg["ckpt_prefix"]
    ckpt_dir = Path(cfg["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    session_start, session_deadline = cfg["session_start_epoch"], cfg["session_start_epoch"] + 60 * cfg["time_budget_min"]
    total_seconds = 3600 * cfg["total_train_hours"]
    log(f"[D] world={world} device={device} | budget this session {cfg['time_budget_min']} min, "
        f"whole run {cfg['total_train_hours']} h | optimizer={cfg['optimizer']}")

    torch.manual_seed(cfg["seed"])
    random.seed(cfg["seed"] + rank)
    model = build_model(model_cfg).to(device)
    n_params = count_params(model)
    log(f"[D] parameters: {n_params / 1e6:.1f}M")

    # ---- resume ----------------------------------------------------------
    step, tokens_seen, seconds_before, rank_positions, ckpt = 0, 0, 0.0, None, None
    phase = cfg.get("phase_id", "main")        # a new phase id = continue training with a fresh LR schedule
    phase_start_step = 0
    latest = find_latest_checkpoint(prefix, [ckpt_dir] + cfg.get("resume_search_dirs", []), ckpt_dir) if main else None
    if distributed:
        box = [str(latest) if latest else ""]
        dist.broadcast_object_list(box, src=0)
        latest = Path(box[0]) if box[0] else None
    if latest is not None:
        ckpt = torch.load(latest, map_location="cpu", weights_only=False)
        assert ckpt["config"] == model_cfg, "Checkpoint config does not match MODEL_D_CONFIG"
        model.load_state_dict(ckpt["model_state"])
        step, tokens_seen = ckpt["step"], ckpt["tokens_seen"]
        same_session = ckpt.get("session_start_epoch") == session_start
        rank_positions = ckpt.get("data_state", {}).get("rank_positions")
        ckpt_phase = ckpt.get("phase_id", "main")
        if ckpt_phase == phase:
            seconds_before = ckpt["seconds_before_session"] if same_session else ckpt["progress_seconds"]
            phase_start_step = ckpt.get("phase_start_step", 0)
        else:
            seconds_before, phase_start_step = 0.0, step
            log(f"[D] new training phase '{phase}' (previous: '{ckpt_phase}') -> fresh LR schedule from step {step:,}")
        log(f"[D] resumed from {latest.name}: step={step:,} tokens={tokens_seen:,} "
            f"phase progress={seconds_before / total_seconds:.1%} ({'same' if same_session else 'new'} session)")
    else:
        log("[D] no checkpoint found -- starting fresh")

    # ---- micro-batch size ---------------------------------------------------
    shapes = cfg["shapes"]
    Pp, Qp = max(((s[0], s[1]) for s in shapes), key=lambda s: s[1])      # longest decoder = most memory
    if cfg.get("fixed_micro_bs") or not cuda:
        micro_bs, sec_micro = cfg.get("fixed_micro_bs") or 2, None
    else:
        res = _probe_micro_batch(model, cfg, device, Pp, Qp)
        fits = [bs for bs, (_, _, ok) in res.items() if ok]
        if not fits:
            model.grad_ckpt = True
            log("[D] nothing fit without activation checkpointing -> turning it on")
            res = _probe_micro_batch(model, cfg, device, Pp, Qp)
            fits = [bs for bs, (_, _, ok) in res.items() if ok]
        for bs, (peak, sec, ok) in res.items():
            log(f"  micro-batch {bs:3d} x 1024 -> " + (f"{peak:5.2f} GB, {sec:.2f}s, {bs * 1024 / sec:,.0f} tok/s/GPU"
                                                      if peak else "OOM") + ("" if ok else "  [no]"))
        best = max(fits) if fits else 0
        if distributed:
            t = torch.tensor([best], device=device)
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
            best = int(t.item())
        assert best > 0, "No micro-batch fits in memory"
        micro_bs, sec_micro = best, res[best][1]
    accum = max(1, round(cfg["tokens_per_step_target"] / (micro_bs * 1024 * world)))
    tokens_per_step = micro_bs * 1024 * world * accum
    log(f"[D] micro-batch {micro_bs} x 1024 tokens x {world} GPU(s) x accum {accum} = {tokens_per_step:,} tokens/step"
        + (f" | probe ~{micro_bs * 1024 * world / sec_micro:,.0f} tok/s" if sec_micro else ""))

    optims = build_optimizers(model, cfg["optimizer"], cfg["peak_lr"], cfg["weight_decay"])
    scaler = make_grad_scaler(cuda)
    if ckpt is not None:
        if ckpt.get("optimizer_kind") == cfg["optimizer"]:
            for o, s in zip(optims, ckpt["optim_states"]):
                o.load_state_dict(s)
            scaler.load_state_dict(ckpt["scaler_state"])
        else:
            log("[D] optimizer changed since the checkpoint -> fresh optimizer state")
        model.grad_ckpt = model.grad_ckpt or ckpt.get("grad_ckpt", False)
        del ckpt
    train_model = model
    if distributed:
        from torch.nn.parallel import DistributedDataParallel as DDP
        train_model = DDP(model, device_ids=[local_rank] if cuda else None, gradient_as_bucket_view=True)

    positions = rank_positions[rank] if rank_positions and rank < len(rank_positions) else None
    files = sorted(Path(cfg["data_dir"]).glob("clean_*.txt"))
    assert files, "No clean_*.txt files in the data folder"
    stream = WindowStream(files, 1024, rank, world, positions)
    data_iter = iter(stream)
    log(f"[D] training files: {[p.name for p in files]} | shapes (prompt, continuation, weight): {shapes}")

    def save_checkpoint(reason):
        gathered = [None] * world
        if distributed:
            dist.all_gather_object(gathered, dict(stream.positions))
        else:
            gathered = [dict(stream.positions)]
        path = None
        if main:
            now = time.time()
            state = {
                "model_state": model.state_dict(), "optim_states": [o.state_dict() for o in optims],
                "optimizer_kind": cfg["optimizer"], "scaler_state": scaler.state_dict(), "step": step,
                "tokens_seen": tokens_seen, "config": model_cfg, "arch_id": ARCH_ID, "run_id": cfg["run_id"],
                "session_start_epoch": session_start, "seconds_before_session": seconds_before,
                "progress_seconds": seconds_before + (now - session_start), "grad_ckpt": model.grad_ckpt,
                "phase_id": phase, "phase_start_step": phase_start_step,
                "data_state": {"rank_positions": gathered, "world": world},
                "train_cfg": {k: cfg[k] for k in ("peak_lr", "shapes", "optimizer", "total_train_hours")},
                "reason": reason,
            }
            path = ckpt_dir / f"{prefix}_step{step}_tokens{tokens_seen}.pt"
            tmp = path.with_name(path.name + ".tmp")
            torch.save(state, tmp)
            tmp.replace(path)
            prune_old_checkpoints(ckpt_dir, prefix, keep=2)
        if distributed:
            dist.barrier()
        return path

    # ---- loop --------------------------------------------------------------
    train_model.train()
    t_loop, last_ckpt, last_log_t, last_log_tokens = time.time(), time.time(), time.time(), tokens_seen
    start_tokens = tokens_seen
    bad_streak, skipped, stop_reason = 0, 0, "unknown"
    ce_log, n_log = torch.zeros((), device=device), 0
    flags = torch.zeros(3, device=device)
    while True:
        if main:
            now = time.time()
            progress = (seconds_before + (now - session_start)) / total_seconds
            stop = 1.0 if (progress >= 1.0 or now >= session_deadline) else 0.0
            do_ckpt = 1.0 if (now - last_ckpt) / 60 >= cfg["checkpoint_every_min"] else 0.0
            flags = torch.tensor([progress, stop, do_ckpt], device=device)
        if distributed:
            dist.broadcast(flags, src=0)
        progress, stop, do_ckpt = float(flags[0]), bool(flags[1] > 0.5), bool(flags[2] > 0.5)
        if stop:
            stop_reason = "schedule complete" if progress >= 1.0 else "session time budget reached"
            break
        lr = cfg["peak_lr"] * lr_factor(step - phase_start_step, progress, cfg["warmup_steps"], cfg["cooldown_frac"], cfg["min_lr_frac"])
        for o in optims:
            for g in o.param_groups:
                g["lr"] = lr
            o.zero_grad(set_to_none=True)
        for micro in range(accum):
            P, Q = choose_shape(shapes, cfg["seed"], step, micro)
            enc, dec_in, tgt = make_example_batch([next(data_iter) for _ in range(micro_bs)], P, Q, device)
            sync_ctx = train_model.no_sync() if (distributed and micro < accum - 1) else contextlib.nullcontext()
            with sync_ctx:
                ce, total = loss_from(train_model(enc, dec_in, targets=tgt), cfg["z_loss_coef"])
                scaler.scale(total / accum).backward()
            ce_log += ce.detach() / accum
            tokens_seen += micro_bs * 1024 * world
        for o in optims:
            scaler.unscale_(o)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]))
        for o in optims:
            scaler.step(o)
        scaler.update()
        n_log += 1
        step += 1
        if math.isfinite(gnorm):
            bad_streak = 0
        else:
            bad_streak += 1
            skipped += 1
            if bad_streak >= 50:
                stop_reason = "50 non-finite steps in a row"
                break
        if distributed and step % cfg["sync_every"] == 0:
            for p in model.parameters():                     # remove any tiny drift between GPUs
                dist.broadcast(p.data, src=0)
        if step % cfg["log_every"] == 0 or step == 1:
            if distributed:
                dist.all_reduce(ce_log)
                ce_log /= world
            now = time.time()
            tps = (tokens_seen - last_log_tokens) / max(1e-6, now - last_log_t)
            log(f"  step {step:6d}  loss={ce_log.item() / max(1, n_log):.4f}  lr={lr:.2e}  gnorm={gnorm:.2f}  "
                f"tokens={tokens_seen:,}  tok/s={tps:,.0f}  elapsed={(now - t_loop) / 60:.1f} min  "
                f"progress={progress:.1%}  [{datetime.datetime.now():%H:%M:%S}]")
            ce_log.zero_()
            n_log, last_log_t, last_log_tokens = 0, now, tokens_seen
        if do_ckpt:
            path = save_checkpoint("periodic")
            last_ckpt = time.time()
            if main:
                print(f"  [checkpoint saved -> {path.name}]", flush=True)
                if cfg["push"]:
                    push_checkpoint_dataset(cfg["output_dir"], cfg["dataset_id"],
                                            f"{cfg['run_id']}: step={step}, tokens={tokens_seen:,} (in progress)")
            if distributed:
                dist.barrier()
    path = save_checkpoint(stop_reason)
    loop_min = (time.time() - t_loop) / 60
    summary = {"stop_reason": stop_reason, "step": step, "tokens_seen": tokens_seen,
               "session_tokens": tokens_seen - start_tokens, "loop_minutes": round(loop_min, 1),
               "tokens_per_sec": round((tokens_seen - start_tokens) / max(1e-6, loop_min * 60), 1),
               "world": world, "micro_bs": micro_bs, "accum": accum, "tokens_per_step": tokens_per_step,
               "skipped_steps": skipped, "grad_ckpt": model.grad_ckpt, "n_params": n_params, "phase_id": phase,
               "phase_start_step": phase_start_step,
               "final_checkpoint": path.name if path else None, "progress": round(progress, 4),
               "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if cuda else 0.0}
    if main:
        json.dump(summary, open(Path(cfg["output_dir"]) / "train_summary.json", "w"), indent=1)
        log(f"[D] stopped ({stop_reason}) at step={step:,}, tokens={tokens_seen:,} after {loop_min:.1f} min "
            f"-> {path.name}")
    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    if stop_reason.startswith("50 non-finite"):
        sys.exit(3)


if __name__ == "__main__":
    train_main(sys.argv[1])


In [ ]:
%%writefile /kaggle/working/c_model.py
"""c_model.py -- Model C (prefix-LM) classes, copied from p1_07_model_C.ipynb."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

decode = None          # only used by C's text generation, not needed here


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True




In [ ]:
%%writefile /kaggle/working/cj_lib.py
"""cj_lib.py -- Phase 4 (Court Judgment Prediction, CJPE) library shared by both notebooks.

Pipeline (every experiment):
  case text -> Phase-1 tokens -> 512-token chunks with 64-token overlap (the first chunk
  plus up to MAX_CHUNKS-1 chunks from the end of the case).
  Stage 1  ADAPT   Phase-1 backbone + adaptation method + attention pooling + 1 logit,
                   trained on ONE chunk per case per step (the last chunk half of the time,
                   otherwise a random chunk), labelled with the case outcome, on 80% of train.
  Stage 2  ENCODE  the adapted backbone turns every chunk of the held-out 20% of train,
                   dev and test into a vector (cached in memory).
  Stage 3  READ    small aggregators over the chunk vectors + chunk positions, trained on the
                   held-out 20% only (the encoder never trained on those cases, so no stacking
                   leakage); compared on a validation slice of that 20%. The long-document and
                   section study (head / middle / end / whole case; mean, max, attention, GRU,
                   transformer) costs seconds here.
  CALIBRATE        decision threshold fitted on dev, used only if a 2-fold dev check agrees.

Methods (own code): LoRMA (Bihany et al. 2025, additive-identity form), AdaLoRA (Zhang et al.
2023), BitFit (Ben Zaken et al. 2022), gradual unfreezing with layer-wise learning rates
(Howard & Ruder 2018), full fine-tuning.
Distillation: hierarchical student (C cut to its first layers, reads the last 4 chunks) learns
gold labels + the teacher ensemble's probabilities + the teachers' chunk attention
("where to look").
"""
import csv
import json
import math
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CHUNK = 512
STRIDE = 448                     # 64-token overlap
MAX_CHUNKS = 16                  # first chunk + 15 from the end
MAX_TOKENS = CHUNK + (CHUNK + (MAX_CHUNKS - 2) * STRIDE)     # tokens kept per case
KD_LAST = 4                      # chunks the student reads


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_json_any(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]


def parse_cases(obj):
    """Accepts a list of case dicts, a {id: case} dict, or {"data": [...]}.
    -> list of {"id": str, "raw_id": as given, "text": str, "label": 0/1 or None}."""
    if isinstance(obj, dict):
        for k in ("data", "cases", "documents", "examples", "items"):
            if k in obj and isinstance(obj[k], list):
                return parse_cases(obj[k])
        items = []
        for k, v in obj.items():
            items.append(dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "text": v})
        return parse_cases(items)
    out = []
    for i, v in enumerate(obj):
        raw = v.get("id", v.get("case_id", v.get("name", i)))
        text = v.get("text", v.get("case_text", v.get("document", v.get("judgment", ""))))
        if isinstance(text, list):
            text = "\n".join(str(t) for t in text)
        lab = v.get("label", v.get("labels", v.get("decision", v.get("outcome"))))
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        if lab is not None:
            lab = int(float(lab))
            assert lab in (0, 1), f"case {raw}: label {lab} is not 0/1"
        out.append({"id": str(raw), "raw_id": raw, "text": " ".join(str(text or "").split()), "label": lab})
    return out


def windows(n, size, stride):
    """Start/end pairs covering 0..n with overlapping windows (last one ends at n)."""
    if n <= size:
        return [(0, n)]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return [(s, s + size) for s in starts]


def case_chunks(toks_len, n_orig, cut):
    """Chunk spans over the kept tokens: (start, end, start_frac, end_frac) where the
    fractions are positions in the ORIGINAL case (0 = first token, 1 = last)."""
    def orig(i):
        return i if (not cut or i < CHUNK) else n_orig - (toks_len - i)
    if cut:
        spans = [(0, CHUNK)] + [(s + CHUNK, e + CHUNK) for s, e in windows(toks_len - CHUNK, CHUNK, STRIDE)]
    else:
        spans = windows(toks_len, CHUNK, STRIDE)
    return [(s, e, orig(s) / n_orig, (orig(e - 1) + 1) / n_orig) for s, e in spans]


def tokenize_cases(cases, encode_batch, fallback_id, batch=256, log=print):
    """encode_batch(list of str) -> list of token-id lists. Keeps the first CHUNK tokens and
    the last MAX_TOKENS - CHUNK tokens of long cases."""
    out, t0 = [], time.time()
    for i in range(0, len(cases), batch):
        part = cases[i:i + batch]
        toks = encode_batch([c["text"] for c in part])
        for c, t in zip(part, toks):
            t = list(t) or [fallback_id]
            n = len(t)
            cut = n > MAX_TOKENS
            kept = t[:CHUNK] + t[n - (MAX_TOKENS - CHUNK):] if cut else t
            arr = np.asarray(kept, dtype=np.int64)
            assert arr.max() < 65536, "token id does not fit uint16"
            out.append({"id": c["id"], "raw_id": c["raw_id"], "label": c["label"], "toks": arr.astype(np.uint16),
                        "n_tok": n, "cut": cut, "chunks": case_chunks(len(kept), n, cut)})
        if (i // batch) % 40 == 0:
            log(f"  tokenized {min(i + batch, len(cases))}/{len(cases)} ({time.time() - t0:.0f}s)")
    return out


def split_train(cases, frac_agg=0.2, frac_val=0.15, seed=7):
    """Stratified split of train into adapt (stage 1) / agg_train / agg_val (stage 3)."""
    rng = random.Random(seed)
    adapt, agg_tr, agg_val = [], [], []
    for lab in (0, 1):
        ids = [c["id"] for c in cases if c["label"] == lab]
        rng.shuffle(ids)
        n_agg = int(round(frac_agg * len(ids)))
        n_val = int(round(frac_val * n_agg))
        agg_val += ids[:n_val]
        agg_tr += ids[n_val:n_agg]
        adapt += ids[n_agg:]
    return {"adapt": sorted(adapt), "agg_train": sorted(agg_tr), "agg_val": sorted(agg_val)}


# ---------------------------------------------------------------------------
# Metrics and calibration
# ---------------------------------------------------------------------------
def binary_report(y, pred):
    y, p = np.asarray(y, int), np.asarray(pred, int)
    rep = {}
    for c in (0, 1):
        tp = int(((p == c) & (y == c)).sum())
        fp = int(((p == c) & (y != c)).sum())
        fn = int(((p != c) & (y == c)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        rep[c] = (prec, rec, 2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return {"macro_f1": round((rep[0][2] + rep[1][2]) / 2, 4),
            "macro_precision": round((rep[0][0] + rep[1][0]) / 2, 4),
            "macro_recall": round((rep[0][1] + rep[1][1]) / 2, 4),
            "f1_0": round(rep[0][2], 4), "f1_1": round(rep[1][2], 4),
            "precision_1": round(rep[1][0], 4), "recall_1": round(rep[1][1], 4),
            "accuracy": round(float((y == p).mean()) if len(y) else 0.0, 4)}


def macro_f1(y, pred):
    return binary_report(y, pred)["macro_f1"]


def roc_auc(y, s):
    y, s = np.asarray(y, int), np.asarray(s, float)
    n1, n0 = int(y.sum()), int((1 - y).sum())
    if n1 == 0 or n0 == 0:
        return 0.5
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s))
    ranks[order] = np.arange(1, len(s) + 1)
    ss = s[order]                                  # average ranks over ties
    i = 0
    while i < len(ss):
        j = i
        while j + 1 < len(ss) and ss[j + 1] == ss[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = (i + j + 2) / 2
        i = j + 1
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def best_threshold(logits, y):
    logits = np.asarray(logits, float)
    cands = np.unique(np.concatenate([[0.0], np.quantile(logits, np.linspace(0.02, 0.98, 97))]))
    scores = [macro_f1(y, logits > t) for t in cands]
    k = int(np.argmax(scores))
    return float(cands[k]), scores[k]


def calibrate(logits, y):
    """Threshold for macro-F1. Default 0 (the class-weighted loss already targets balanced
    classes). A dev-fitted threshold is used only if fitting on one half of dev and scoring
    the other half (both ways) beats the default."""
    logits, y = np.asarray(logits, float), np.asarray(y, int)
    f1_default = macro_f1(y, logits > 0)
    idx = np.arange(len(y))
    a = np.zeros(len(y), bool)
    for lab in (0, 1):                             # stratified halves
        ids = idx[y == lab]
        a[ids[::2]] = True
    pred = (logits > 0).astype(int)
    if a.any() and (~a).any():
        ta, _ = best_threshold(logits[a], y[a])
        tb, _ = best_threshold(logits[~a], y[~a])
        pred[~a] = logits[~a] > ta
        pred[a] = logits[a] > tb
    f1_cv = macro_f1(y, pred)
    t_all, f1_fit = best_threshold(logits, y)
    use = f1_cv > f1_default
    return (t_all if use else 0.0), {"dev_f1_threshold0": f1_default, "dev_f1_threshold_2fold": f1_cv,
                                     "dev_f1_threshold_fitted_on_dev": f1_fit, "threshold_used": round(t_all if use else 0.0, 4),
                                     "threshold_fitted": use}


# ---------------------------------------------------------------------------
# Adaptation methods
# ---------------------------------------------------------------------------
LINEAR_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


class LoRMALinear(nn.Module):
    """LoRMA, additive-identity form: y = ((s B A) + I) W0 x = h + s B A h with h = W0 x.
    B = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


class SVDLinear(nn.Module):
    """AdaLoRA: y = W0 x + s P diag(E) Q x. E (the 'singular values') starts at 0, so the
    layer starts as the Phase-1 layer; pruning a triplet sets its E entry to 0."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.Q) * self.E, self.P) * self.scale


def _replace(module, name, new):
    parent = module
    parts = name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def apply_method(model, method, r=16, alpha=32):
    """Freeze the backbone, then open / insert what the method trains. Returns the number of
    backbone-side trainable parameters (the gradual-unfreezing controller opens blocks later)."""
    for p in model.parameters():
        p.requires_grad_(False)
    if method == "full":
        for p in model.parameters():
            p.requires_grad_(True)
    elif method == "bitfit":
        n = 0
        for name, p in model.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad_(True)
                n += 1
        assert n, "BitFit needs bias terms; this backbone has none (use C)."
    elif method in ("lorma", "adalora"):
        names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in LINEAR_TARGETS]
        assert names, "no target linear layers found"
        for n in names:
            lin = model.get_submodule(n)
            _replace(model, n, LoRMALinear(lin, r, alpha) if method == "lorma" else SVDLinear(lin, r, alpha))
    elif method != "unfreeze":
        raise ValueError(method)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class AdaLoRA:
    """Importance-based rank allocation (Zhang et al. 2023). Importance of each weight =
    smoothed |w * grad| (EMA, beta1) times its smoothed uncertainty (EMA of the deviation,
    beta2); a triplet's score = score(E_i) + mean score of P[:, i] + mean score of Q[i, :].
    The total rank budget falls from r_init to r_target per layer on a cubic schedule between
    progress t_init and 1 - t_final, then stays fixed. Orthogonality penalty on P and Q."""

    def __init__(self, model, r_init, r_target, t_init=0.15, t_final=0.25, every=20, beta1=0.85, beta2=0.85, orth=0.1):
        self.mods = [(n, m) for n, m in model.named_modules() if isinstance(m, SVDLinear)]
        self.b0, self.bT = r_init * len(self.mods), r_target * len(self.mods)
        self.t_init, self.t_final, self.every = t_init, t_final, every
        self.beta1, self.beta2, self.orth = beta1, beta2, orth
        self.ipt, self.unc, self.final_mask = {}, {}, None

    def orth_loss(self):
        reg = 0.0
        for _, m in self.mods:
            r = m.E.shape[0]
            eye = torch.eye(r, device=m.P.device)
            reg = reg + torch.norm(m.P.float().t() @ m.P.float() - eye, p="fro") + torch.norm(m.Q.float() @ m.Q.float().t() - eye, p="fro")
        return self.orth * reg / max(1, len(self.mods))

    @torch.no_grad()
    def update_importance(self):
        for n, m in self.mods:
            for key, p in (("P", m.P), ("E", m.E), ("Q", m.Q)):
                if p.grad is None:
                    continue
                s = (p * p.grad).abs()
                k = (n, key)
                if k not in self.ipt:
                    self.ipt[k], self.unc[k] = torch.zeros_like(s), torch.zeros_like(s)
                self.ipt[k].mul_(self.beta1).add_((1 - self.beta1) * s)
                self.unc[k].mul_(self.beta2).add_((1 - self.beta2) * (s - self.ipt[k]).abs())

    def budget(self, p):
        if p < self.t_init:
            return self.b0
        if p >= 1 - self.t_final:
            return self.bT
        x = (p - self.t_init) / (1 - self.t_init - self.t_final)
        return int(self.bT + (self.b0 - self.bT) * (1 - x) ** 3)

    def _scores(self):
        out = []
        for n, m in self.mods:
            if (n, "E") not in self.ipt:
                out.append(torch.zeros_like(m.E))
                continue
            sc = lambda k: self.ipt[(n, k)] * self.unc[(n, k)]
            out.append(sc("E") + sc("P").mean(0) + sc("Q").mean(1))
        return out

    @torch.no_grad()
    def mask(self, progress, step):
        if progress < self.t_init:
            return
        if progress >= 1 - self.t_final:
            if self.final_mask is None:
                self.final_mask = self._prune(self.bT)
            for (_, m), keep in zip(self.mods, self.final_mask):
                m.E.mul_(keep)
            return
        if step % self.every == 0:
            self._prune(self.budget(progress))

    def _prune(self, budget):
        scores = self._scores()
        allv = torch.cat([s.flatten().float() for s in scores])
        if budget >= allv.numel():
            return [torch.ones_like(m.E) for _, m in self.mods]
        thr = torch.kthvalue(allv, allv.numel() - budget).values
        keeps = []
        for (_, m), s in zip(self.mods, scores):
            keep = (s.float() > thr).to(m.E.dtype)
            m.E.mul_(keep)
            keeps.append(keep)
        return keeps

    def ranks(self):
        return {n: int((m.E != 0).sum()) for n, m in self.mods}


class Unfreezer:
    """Gradual unfreezing from the top: at progress t, the top k blocks train
    (stages = ((t0, k0), (t1, k1), ...)); the final norm always trains."""

    def __init__(self, blocks, always, stages, log=print):
        self.blocks, self.stages, self.log, self.n_open, self.events = list(blocks), stages, log, 0, []
        for p in always:
            p.requires_grad_(True)

    def update(self, progress):
        n = max(k for t, k in self.stages if progress >= t)
        if n != self.n_open:
            for b in self.blocks[-n:]:
                for p in b.parameters():
                    p.requires_grad_(True)
            self.n_open = n
            self.events.append({"progress": round(progress, 3), "top_blocks_training": n})
            self.log(f"  gradual unfreezing: top {n} of {len(self.blocks)} blocks now train (progress {progress:.2f})")


# ---------------------------------------------------------------------------
# Encoders and models
# ---------------------------------------------------------------------------
def blocks_of(kind, model):
    return model.enc_blocks if kind == "D" else model.blocks


def final_norm(kind, model):
    return model.enc_norm if kind == "D" else model.norm_f


class AttnPool(nn.Module):
    """Attention pooling over tokens; the query starts at 0, i.e. as mean pooling."""

    def __init__(self, d):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(d))

    def forward(self, h, mask):
        s = (h @ self.q) / math.sqrt(h.shape[-1])
        w = torch.softmax(s.masked_fill(~mask, -1e4), -1)
        return (w.unsqueeze(-1) * h).sum(1)


class ChunkEncoder(nn.Module):
    """token chunk [B, T] (right-padded) -> chunk vector [B, d]."""

    def __init__(self, kind, model, d):
        super().__init__()
        self.kind, self.m, self.d = kind, model, d
        self.pool = AttnPool(d)

    def states(self, ids, mask):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=ids.is_cuda):
            if self.kind == "D":
                h = self.m.encode(ids, mask)
            elif self.kind == "C":
                h = self.m.encode(ids, mode="prefix", prefix_len=mask.sum(1))
            else:                                   # A, B: left to right
                h = self.m.encode(ids)
        return h.float()

    def forward(self, ids, mask):
        return self.pool(self.states(ids, mask), mask)


class ChunkModel(nn.Module):
    def __init__(self, encoder, d, drop=0.1):
        super().__init__()
        self.enc, self.drop, self.out = encoder, nn.Dropout(drop), nn.Linear(d, 1)

    def forward(self, ids, mask):
        v = self.enc(ids, mask)
        return self.out(self.drop(v)).squeeze(-1), v


class Aggregator(nn.Module):
    """Case-level reader over chunk vectors X [B, n, d], positions P [B, n, 4]
    (start, end, is_first, is_last), mask M [B, n]. Returns (logit [B], chunk weights or None)."""

    KINDS = ("tail", "head", "mean", "max", "attn", "gru_attn", "transformer")

    def __init__(self, d, kind, h=256, drop=0.2):
        super().__init__()
        self.kind = kind
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, h), nn.GELU())
        self.pos = nn.Linear(4, h)
        if kind == "gru_attn":
            self.gru = nn.GRU(h, h // 2, batch_first=True, bidirectional=True)
        if kind == "transformer":
            layer = nn.TransformerEncoderLayer(h, 4, 2 * h, drop, batch_first=True, norm_first=True)
            self.tr = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
            self.cls = nn.Parameter(torch.zeros(1, 1, h))
        self.score = nn.Linear(h, 1)
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(h, 1))

    def forward(self, X, P, M):
        x = self.inp(X) + self.pos(P)
        B, n, _ = x.shape
        lens = M.sum(1)
        w = None
        if self.kind == "tail":
            z = x[torch.arange(B, device=x.device), lens - 1]
        elif self.kind == "head":
            z = x[:, 0]
        elif self.kind == "mean":
            m = M.unsqueeze(-1).float()
            z = (x * m).sum(1) / m.sum(1).clamp(min=1)
        elif self.kind == "max":
            z = x.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
        elif self.kind == "transformer":
            t = torch.cat([self.cls.expand(B, -1, -1), x], 1)
            pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=x.device), ~M], 1)
            z = self.tr(t, src_key_padding_mask=pad)[:, 0]
        else:
            if self.kind == "gru_attn":
                packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
                x = nn.utils.rnn.pad_packed_sequence(self.gru(packed)[0], batch_first=True, total_length=n)[0]
            s = self.score(x).squeeze(-1).masked_fill(~M, -1e4)
            w = torch.softmax(s, -1)
            z = (w.unsqueeze(-1) * x).sum(1)
        return self.out(z).squeeze(-1), w


def pad_tokens(seqs, pad_id, device):
    L = max(1, max(len(s) for s in seqs))
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.from_numpy(np.asarray(s, dtype=np.int64))
        mask[i, :len(s)] = True
    return ids.to(device), mask.to(device)


def pos_feats(case, j):
    s, e, fs, fe = case["chunks"][j]
    return [fs, fe, float(j == 0), float(j == len(case["chunks"]) - 1)]


def region_select(case, region):
    """Chunk indices whose centre lies in the region ('begin' < 1/3, 'middle', 'end' > 2/3);
    if none does, the chunk whose centre is closest to the region."""
    lo, hi = {"begin": (0, 1 / 3), "middle": (1 / 3, 2 / 3), "end": (2 / 3, 1.0001)}[region]
    cent = [(fs + fe) / 2 for _, _, fs, fe in case["chunks"]]
    sel = [j for j, c in enumerate(cent) if lo <= c < hi]
    if not sel:
        sel = [int(np.argmin([min(abs(c - lo), abs(c - hi)) for c in cent]))]
    return sel


# ---------------------------------------------------------------------------
# Stage 1: adapt the backbone on single chunks
# ---------------------------------------------------------------------------
def lr_mult(p, warm=0.05):
    return min(1.0, p / warm) if p < warm else max(0.1, 1 - (p - warm) / (1 - warm))


def pick_chunk(case, rng):
    n = len(case["chunks"])
    return n - 1 if rng.random() < 0.5 else rng.randrange(n)


def build_net(exp, make_backbone, device, log):
    model, d, kind = make_backbone(exp["model"])
    n_backbone = sum(p.numel() for p in model.parameters())
    method = exp["method"]
    apply_method(model, method, exp.get("r", 16), exp.get("alpha", 32))
    net = ChunkModel(ChunkEncoder(kind, model, d), d).to(device)
    ada = unf = None
    head = list(net.enc.pool.parameters()) + list(net.out.parameters())
    groups = [{"params": head, "lr": exp["lr_head"], "base_lr": exp["lr_head"]}]
    if method == "unfreeze":
        blocks = list(blocks_of(kind, model))
        norm = list(final_norm(kind, model).parameters())
        unf = Unfreezer(blocks, norm, exp["stages"], log)
        groups.append({"params": norm, "lr": exp["lr"], "base_lr": exp["lr"]})
        k_max = max(k for _, k in exp["stages"])
        for i, b in enumerate(reversed(blocks[-k_max:])):        # i = 0 is the top block
            lr = exp["lr"] * exp.get("lr_decay", 0.85) ** i
            groups.append({"params": list(b.parameters()), "lr": lr, "base_lr": lr})
    else:
        bb = [p for p in model.parameters() if p.requires_grad]
        if bb:
            groups.append({"params": bb, "lr": exp["lr"], "base_lr": exp["lr"]})
    if method == "adalora":
        ada = AdaLoRA(net, exp["r"], exp["r_target"], orth=exp.get("orth", 0.1))
    return net, kind, n_backbone, groups, ada, unf


def trainable_count(net, unf, exp):
    if unf is not None:            # what eventually trains: head + final norm + top k_max blocks
        k_max = max(k for _, k in exp["stages"])
        blocks = unf.blocks[-k_max:]
        n = sum(p.numel() for b in blocks for p in b.parameters())
        return n + sum(p.numel() for p in net.parameters() if p.requires_grad and not any(p is q for b in blocks for q in b.parameters()))
    return sum(p.numel() for p in net.parameters() if p.requires_grad)


@torch.no_grad()
def last_chunk_scores(net, cases, pad_id, device, bs):
    net.eval()
    out = []
    for i in range(0, len(cases), bs):
        part = cases[i:i + bs]
        seqs = [c["toks"][c["chunks"][-1][0]:c["chunks"][-1][1]] for c in part]
        ids, mask = pad_tokens(seqs, pad_id, device)
        out += net(ids, mask)[0].float().cpu().tolist()
    return np.array(out)


def train_stage1(exp, data, make_backbone, device, log):
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    net, kind, n_backbone, groups, ada, unf = build_net(exp, make_backbone, device, log)
    pad_id = data["pad_id"]
    by_id = {c["id"]: c for c in data["train"]}
    adapt = [by_id[i] for i in data["split"]["adapt"]]
    val = [by_id[i] for i in data["split"]["agg_val"]]
    y_val = [c["label"] for c in val]
    n_pos = sum(c["label"] for c in adapt)
    pos_weight = torch.tensor((len(adapt) - n_pos) / max(1, n_pos), device=device)
    opt = torch.optim.AdamW([g for g in groups], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mb, accum = exp["micro_bs"], exp["accum"]
    planned = exp["epochs"] * max(1, math.ceil(len(adapt) / mb) // accum)
    budget = exp["budget_min"] * 60
    n_tr = trainable_count(net, unf, exp)
    n_head = sum(p.numel() for p in net.enc.pool.parameters()) + sum(p.numel() for p in net.out.parameters())
    pct = 100 * (n_tr - n_head) / n_backbone
    log(f"[{exp['name']}] backbone {exp['model']} {n_backbone / 1e6:.1f}M | method {exp['method']} | trainable "
        f"{n_tr - n_head:,} in the backbone ({pct:.2f}%) + {n_head:,} pooling/head | adapt cases {len(adapt)} (pos weight "
        f"{float(pos_weight):.2f}) | {planned} optimizer steps planned, budget {exp['budget_min']} min")
    t0, step, micro = time.time(), 0, 0
    st = {"best_auc": -1.0, "best_state": None, "hist": [], "run_loss": 0.0, "n_loss": 0}
    eval_points = [0.25, 0.5, 0.75, 1.0]
    progress, done = 0.0, False
    params_all = [p for g in groups for p in g["params"]]
    tracked = {id(p) for p in params_all}

    def evaluate():
        auc = roc_auc(y_val, last_chunk_scores(net, val, pad_id, device, exp["infer_bs"]))
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run_loss"] / max(1, st["n_loss"]), 4),
                           "val_auc_last_chunk": round(auc, 4), "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | train loss {h['train_loss']:.4f} | "
            f"val AUC (last chunk) {auc:.4f} | {h['minutes']} min")
        st["run_loss"], st["n_loss"] = 0.0, 0
        # AdaLoRA: only checkpoints whose rank budget is final can be chosen
        eligible = ada is None or progress >= 1 - ada.t_final
        if eligible and auc > st["best_auc"]:
            st["best_auc"] = auc
            st["best_state"] = {n: p.detach().cpu().clone() for n, p in net.named_parameters() if id(p) in tracked}

    for epoch in range(exp["epochs"]):
        order = list(range(len(adapt)))
        rng.shuffle(order)
        for i in range(0, len(order), mb):
            net.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            if unf is not None:
                unf.update(progress)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            part = [adapt[k] for k in order[i:i + mb]]
            seqs = []
            for c in part:
                j = pick_chunk(c, rng)
                s, e = c["chunks"][j][:2]
                seqs.append(c["toks"][s:e])
            ids, mask = pad_tokens(seqs, pad_id, device)
            y = torch.tensor([c["label"] for c in part], dtype=torch.float32, device=device)
            logit = net(ids, mask)[0].float()
            loss = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight)
            total = loss + (ada.orth_loss() if ada is not None else 0.0)
            scaler.scale(total / accum).backward()
            st["run_loss"], st["n_loss"], micro = st["run_loss"] + float(loss.detach()), st["n_loss"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            if ada is not None and all(torch.isfinite(p.grad).all() for p in params_all if p.grad is not None):
                ada.update_importance()
            torch.nn.utils.clip_grad_norm_([p for p in params_all if p.grad is not None], 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            if ada is not None:
                ada.mask(progress, step)
            while eval_points and progress >= eval_points[0]:
                eval_points.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if eval_points or st["best_state"] is None:      # epochs ran out just before the last evaluation point
        progress = 1.0
        if ada is not None:
            ada.mask(progress, step)
        evaluate()
    net.load_state_dict(st["best_state"], strict=False)
    info = {"n_backbone": n_backbone, "n_trainable": n_tr, "trainable_pct": round(pct, 3),
            "stage1_minutes": round((time.time() - t0) / 60, 1), "stage1_steps": step, "stage1_history": st["hist"],
            "best_val_auc_last_chunk": round(st["best_auc"], 4),
            "peak_mem_train_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0}
    if ada is not None:
        info["adalora_ranks"] = ada.ranks()
    if unf is not None:
        info["unfreeze_events"] = unf.events
    return net, info


# ---------------------------------------------------------------------------
# Stage 2: encode chunks
# ---------------------------------------------------------------------------
@torch.no_grad()
def encode_cases(net, cases, pad_id, device, bs, last_k=None):
    """{case id: (vectors [n, d] fp16, stage-1 logits [n], positions [n, 4], chunk index list)}."""
    net.eval()
    items = []
    for ci, c in enumerate(cases):
        js = list(range(len(c["chunks"])))
        if last_k:
            js = js[-last_k:]
        items += [(ci, j) for j in js]
    items.sort(key=lambda t: -(cases[t[0]]["chunks"][t[1]][1] - cases[t[0]]["chunks"][t[1]][0]))
    vec, lg = {}, {}
    for i in range(0, len(items), bs):
        part = items[i:i + bs]
        seqs = [cases[ci]["toks"][cases[ci]["chunks"][j][0]:cases[ci]["chunks"][j][1]] for ci, j in part]
        ids, mask = pad_tokens(seqs, pad_id, device)
        logit, v = net(ids, mask)
        v, logit = v.half().cpu(), logit.float().cpu()
        for k, (ci, j) in enumerate(part):
            vec[(ci, j)], lg[(ci, j)] = v[k], float(logit[k])
    out = {}
    for ci, c in enumerate(cases):
        js = list(range(len(c["chunks"])))
        if last_k:
            js = js[-last_k:]
        out[c["id"]] = (torch.stack([vec[(ci, j)] for j in js]), np.array([lg[(ci, j)] for j in js]),
                        np.array([pos_feats(c, j) for j in js], dtype=np.float32), js)
    return out


# ---------------------------------------------------------------------------
# Stage 3: aggregators
# ---------------------------------------------------------------------------
def stack(enc, ids, cases_by_id, region=None, last_k=None):
    rows = []
    for i in ids:
        V, _, P, js = enc[i]
        sel = list(range(len(js)))
        if region:
            allowed = set(region_select(cases_by_id[i], region))
            sel = [k for k, j in enumerate(js) if j in allowed] or [len(js) - 1]
        if last_k:
            sel = sel[-last_k:]
        rows.append((V[sel].float(), torch.from_numpy(P[sel])))
    n = max(r[0].shape[0] for r in rows)
    d = rows[0][0].shape[1]
    X = torch.zeros(len(rows), n, d)
    Pm = torch.zeros(len(rows), n, 4)
    M = torch.zeros(len(rows), n, dtype=torch.bool)
    for b, (v, p) in enumerate(rows):
        X[b, :len(v)], Pm[b, :len(v)], M[b, :len(v)] = v, p, True
    return X, Pm, M


@torch.no_grad()
def agg_predict(agg, X, P, M, device, bs=256):
    agg.eval()
    logits, weights = [], []
    for i in range(0, len(X), bs):
        lg, w = agg(X[i:i + bs].to(device), P[i:i + bs].to(device), M[i:i + bs].to(device))
        logits.append(lg.float().cpu())
        weights.append(w.float().cpu() if w is not None else None)
    W = None if weights[0] is None else [w for part in weights for w in part]
    return torch.cat(logits).numpy(), W


def train_aggregator(kind, Xt, Pt, Mt, yt, Xv, Pv, Mv, yv, device, seed, epochs=40, patience=6, lr=1e-3, bs=64):
    torch.manual_seed(seed)
    agg = Aggregator(Xt.shape[-1], kind).to(device)
    opt = torch.optim.AdamW(agg.parameters(), lr=lr, weight_decay=0.01)
    y = torch.tensor(yt, dtype=torch.float32)
    pw = torch.tensor((len(yt) - sum(yt)) / max(1, sum(yt)), device=device)
    best, best_state, bad = -1.0, None, 0
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        agg.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            lg, _ = agg(Xt[b].to(device), Pt[b].to(device), Mt[b].to(device))
            loss = F.binary_cross_entropy_with_logits(lg, y[b].to(device), pos_weight=pw)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        auc = roc_auc(yv, agg_predict(agg, Xv, Pv, Mv, device)[0])
        if auc > best + 1e-5:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in agg.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    agg.load_state_dict(best_state)
    return agg, best


def attention_by_position(weights, enc, ids, bins=10):
    """Mean share of attention per decile of case position (chunk centre)."""
    acc = np.zeros(bins)
    for w, i in zip(weights, ids):
        P = enc[i][2]
        for k in range(len(P)):
            c = min(bins - 1, int((P[k, 0] + P[k, 1]) / 2 * bins))
            acc[c] += float(w[k])
    return (acc / max(1, len(ids))).round(4).tolist()


def stage3(exp, data, enc, device, log, out_dir, kd_enc=None):
    """Aggregator study + final case predictions. enc covers agg_train, agg_val, dev, test."""
    seed = exp["seed"]
    cases = {c["id"]: c for split in ("train", "dev", "test") for c in data[split]}
    tr_ids, va_ids = data["split"]["agg_train"], data["split"]["agg_val"]
    dev_ids, test_ids = [c["id"] for c in data["dev"]], [c["id"] for c in data["test"]]
    yt, yv, yd = [cases[i]["label"] for i in tr_ids], [cases[i]["label"] for i in va_ids], [cases[i]["label"] for i in dev_ids]
    rows, models = [], {}

    def report(name, val_scores, dev_scores, note=""):
        r = {"reader": name, "val_auc": round(roc_auc(yv, val_scores), 4), "dev_auc": round(roc_auc(yd, dev_scores), 4),
             "dev_macro_f1_at_0": macro_f1(yd, np.asarray(dev_scores) > 0), "note": note}
        rows.append(r)
        log(f"[{exp['name']}]   {name:28s} val AUC {r['val_auc']:.4f} | dev AUC {r['dev_auc']:.4f} | dev macro-F1 {r['dev_macro_f1_at_0']:.4f}")
        return r

    log(f"[{exp['name']}] stage 3: case readers (trained on {len(tr_ids)} held-out train cases, chosen on {len(va_ids)})")
    for name, f in (("stage1 last chunk only", lambda i: enc[i][1][-1]), ("stage1 mean of chunk logits", lambda i: enc[i][1].mean())):
        report(name, [f(i) for i in va_ids], [f(i) for i in dev_ids], "no training")
    full = {s: stack(enc, ids, cases) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids), ("test", test_ids))}
    for kind in Aggregator.KINDS:
        agg, _ = train_aggregator(kind, *full["tr"], yt, *full["va"], yv, device, seed)
        models[kind] = agg
        report(kind, agg_predict(agg, *full["va"], device)[0], agg_predict(agg, *full["dev"], device)[0], "whole case")
    for region in ("begin", "middle", "end"):
        st = {s: stack(enc, ids, cases, region=region) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids))}
        agg, _ = train_aggregator("attn", *st["tr"], yt, *st["va"], yv, device, seed)
        report(f"attn on {region} third only", agg_predict(agg, *st["va"], device)[0], agg_predict(agg, *st["dev"], device)[0], "section study")
    st = {s: stack(enc, ids, cases, last_k=KD_LAST) for s, ids in (("tr", tr_ids), ("va", va_ids), ("dev", dev_ids))}
    agg, _ = train_aggregator("attn", *st["tr"], yt, *st["va"], yv, device, seed)
    report(f"attn on last {KD_LAST} chunks", agg_predict(agg, *st["va"], device)[0], agg_predict(agg, *st["dev"], device)[0], "section study")

    best = max([r for r in rows if r["note"] == "whole case"], key=lambda r: r["val_auc"])
    chosen = models[best["reader"]]
    dev_logits = agg_predict(chosen, *full["dev"], device)[0]
    test_logits = agg_predict(chosen, *full["test"], device)[0]
    thr, cal = calibrate(dev_logits, yd)
    dev_pred = (dev_logits > thr).astype(int)
    test_pred = (test_logits > thr).astype(int)
    rep = binary_report(yd, dev_pred)
    log(f"[{exp['name']}] chosen reader: {best['reader']} (best val AUC {best['val_auc']:.4f}) | dev macro-F1 "
        f"{cal['dev_f1_threshold0']:.4f} at threshold 0, {cal['dev_f1_threshold_2fold']:.4f} with a 2-fold dev threshold "
        f"-> using threshold {thr:.3f} | final dev macro-F1 {rep['macro_f1']:.4f}")
    _, wdev = agg_predict(models["attn"], *full["dev"], device)
    att_pos = attention_by_position(wdev, enc, dev_ids)
    with open(Path(out_dir) / "readers.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    with open(Path(out_dir) / "dev_cases.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "label", "pred", "logit", "n_tokens", "n_chunks", "outcome"])
        for i, y, p, lg in zip(dev_ids, yd, dev_pred, dev_logits):
            c = cases[i]
            w.writerow([i, y, int(p), round(float(lg), 4), c["n_tok"], len(c["chunks"]),
                        "TP" if y and p else "TN" if not y and not p else "FP" if p else "FN"])
    errors = error_summary(yd, dev_pred, [cases[i]["n_tok"] for i in dev_ids])
    res = {"readers": rows, "chosen_reader": best["reader"], "calibration": cal, "dev_report": rep,
           "dev_auc": round(roc_auc(yd, dev_logits), 4), "attention_by_position_dev": att_pos, "errors_by_length": errors,
           "dev_logits": {i: float(v) for i, v in zip(dev_ids, dev_logits)},
           "test_logits": {i: float(v) for i, v in zip(test_ids, test_logits)}, "threshold": thr}
    if kd_enc is not None:                         # distillation targets on every train case's last chunks
        attn = models["attn"]
        targets = {}
        tr_all = [c["id"] for c in data["train"]]
        src = {i: (kd_enc[i] if i in kd_enc else enc[i]) for i in tr_all}
        for k in range(0, len(tr_all), 2048):
            ids = tr_all[k:k + 2048]
            X, P, M = stack(src, ids, cases, last_k=KD_LAST)
            lg, _ = agg_predict(chosen, X, P, M, device)
            _, wa = agg_predict(attn, X, P, M, device)
            for i, l, wv, m in zip(ids, lg, wa, M):
                targets[i] = {"logit": float(l), "attn": wv[:int(m.sum())].tolist()}
        torch.save(targets, Path(out_dir) / "kd_targets.pt")
        log(f"[{exp['name']}] wrote distillation targets for {len(targets)} train cases")
    return res, test_pred


def error_summary(y, pred, lengths):
    y, p, L = np.asarray(y), np.asarray(pred), np.asarray(lengths)
    qs = np.quantile(L, [0.25, 0.5, 0.75])
    out = []
    for k, (lo, hi) in enumerate(((0, qs[0]), (qs[0], qs[1]), (qs[1], qs[2]), (qs[2], np.inf))):
        m = (L >= lo) & (L < hi) if k < 3 else (L >= lo)
        if m.sum() == 0:
            continue
        out.append({"length_quartile": k + 1, "tokens_from": int(lo), "cases": int(m.sum()),
                    "macro_f1": macro_f1(y[m], p[m]), "false_pos": int(((p == 1) & (y == 0) & m).sum()),
                    "false_neg": int(((p == 0) & (y == 1) & m).sum())})
    return out


# ---------------------------------------------------------------------------
# Distillation student: reads the last KD_LAST chunks end to end
# ---------------------------------------------------------------------------
class HierStudent(nn.Module):
    def __init__(self, encoder, d):
        super().__init__()
        self.enc = encoder
        self.agg = Aggregator(d, "attn")

    def forward(self, cases, pad_id, device):
        seqs, owners, P = [], [], []
        for b, c in enumerate(cases):
            js = list(range(len(c["chunks"])))[-KD_LAST:]
            for j in js:
                s, e = c["chunks"][j][:2]
                seqs.append(c["toks"][s:e])
                owners.append(b)
                P.append(pos_feats(c, j))
        ids, mask = pad_tokens(seqs, pad_id, device)
        v = self.enc(ids, mask)
        n = max(owners.count(b) for b in range(len(cases)))
        X = torch.zeros(len(cases), n, v.shape[-1], device=device)
        Pm = torch.zeros(len(cases), n, 4, device=device)
        M = torch.zeros(len(cases), n, dtype=torch.bool, device=device)
        fill = [0] * len(cases)
        Pt = torch.tensor(P, device=device)
        for k, b in enumerate(owners):
            X[b, fill[b]], Pm[b, fill[b]], M[b, fill[b]] = v[k], Pt[k], True
            fill[b] += 1
        return self.agg(X, Pm, M)


def run_student(exp, data, make_backbone, device, log, out_dir, teacher_files):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    model, d, kind = make_backbone(exp["model"])
    apply_method(model, "full")
    n_backbone = sum(p.numel() for p in model.parameters())
    net = HierStudent(ChunkEncoder(kind, model, d), d).to(device)
    pad_id = data["pad_id"]
    teachers = [torch.load(f, weights_only=False) for f in teacher_files]
    T, alpha, beta = exp.get("kd_T", 2.0), exp.get("kd_alpha", 0.0), exp.get("attn_beta", 0.0)
    by_id = {c["id"]: c for c in data["train"]}
    val_ids = set(data["split"]["agg_val"])
    train = [c for c in data["train"] if c["id"] not in val_ids]
    val = [by_id[i] for i in data["split"]["agg_val"]]
    n_pos = sum(c["label"] for c in train)
    pos_weight = torch.tensor((len(train) - n_pos) / max(1, n_pos), device=device)
    enc_params = list(net.enc.m.parameters())
    groups = [{"params": enc_params, "lr": exp["lr"], "base_lr": exp["lr"]},
              {"params": list(net.enc.pool.parameters()) + list(net.agg.parameters()), "lr": exp["lr_head"], "base_lr": exp["lr_head"]}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mb, accum = exp["micro_bs"], exp["accum"]
    planned = exp["epochs"] * max(1, math.ceil(len(train) / mb) // accum)
    budget = exp["budget_min"] * 60
    n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad)
    log(f"[{exp['name']}] student {exp['model']} ({n_backbone / 1e6:.1f}M, first {exp['student_layers']} layers of C) reads the "
        f"last {KD_LAST} chunks | teachers: {exp.get('teacher_names') or 'none (gold labels only)'} | alpha {alpha} T {T} "
        f"attention beta {beta} | {len(train)} cases, {planned} steps planned, budget {exp['budget_min']} min")
    t0, step, micro, progress, done = time.time(), 0, 0, 0.0, False
    st = {"best_auc": -1.0, "best_state": None, "hist": [], "run_loss": 0.0, "n_loss": 0}
    eval_points = [0.25, 0.5, 0.75, 1.0]
    y_val = [c["label"] for c in val]

    def evaluate():
        auc = roc_auc(y_val, student_logits(net, val, pad_id, device, exp["infer_bs"]))
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run_loss"] / max(1, st["n_loss"]), 4),
                           "val_auc": round(auc, 4), "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | loss {h['train_loss']:.4f} | val AUC {auc:.4f} | {h['minutes']} min")
        st["run_loss"], st["n_loss"] = 0.0, 0
        if auc > st["best_auc"]:
            st["best_auc"] = auc
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}

    for epoch in range(exp["epochs"]):
        order = list(range(len(train)))
        rng.shuffle(order)
        for i in range(0, len(order), mb):
            net.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            part = [train[k] for k in order[i:i + mb]]
            logit, w = net(part, pad_id, device)
            logit = logit.float()
            y = torch.tensor([c["label"] for c in part], dtype=torch.float32, device=device)
            loss = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight)
            if teachers and alpha > 0:
                p_t = torch.tensor([np.mean([1 / (1 + math.exp(-t[c["id"]]["logit"] / T)) for t in teachers]) for c in part],
                                   dtype=torch.float32, device=device)
                kd = F.binary_cross_entropy_with_logits(logit / T, p_t) * T * T
                loss = (1 - alpha) * loss + alpha * kd
            if teachers and beta > 0:
                kl = 0.0
                for b, c in enumerate(part):
                    a_t = torch.tensor(np.mean([t[c["id"]]["attn"] for t in teachers], 0), dtype=torch.float32, device=device)
                    a_s = w[b, :len(a_t)].float().clamp(min=1e-6)
                    kl = kl + (a_t * (a_t.clamp(min=1e-6).log() - a_s.log())).sum()
                loss = loss + beta * kl / len(part)
            scaler.scale(loss / accum).backward()
            st["run_loss"], st["n_loss"], micro = st["run_loss"] + float(loss.detach()), st["n_loss"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            while eval_points and progress >= eval_points[0]:
                eval_points.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if eval_points:
        evaluate()
    best_auc, hist = st["best_auc"], st["hist"]
    net.load_state_dict(st["best_state"])
    train_min = round((time.time() - t0) / 60, 1)
    peak_train = round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t1 = time.time()
    dev_logits = student_logits(net, data["dev"], pad_id, device, exp["infer_bs"])
    infer_s = time.time() - t1
    test_logits = student_logits(net, data["test"], pad_id, device, exp["infer_bs"])
    yd = [c["label"] for c in data["dev"]]
    thr, cal = calibrate(dev_logits, yd)
    rep = binary_report(yd, dev_logits > thr)
    info = {"exp": exp, "n_backbone": n_backbone, "n_trainable": n_tr, "trainable_pct": 100.0, "train_minutes": train_min,
            "history": hist, "best_val_auc": round(best_auc, 4), "peak_mem_train_gb": peak_train,
            "peak_mem_infer_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0,
            "infer_cases_per_s": round(len(data["dev"]) / max(1e-6, infer_s), 2), "calibration": cal, "dev_report": rep,
            "dev_auc": round(roc_auc(yd, dev_logits), 4), "chosen_reader": f"student attn over last {KD_LAST} chunks",
            "threshold": thr, "dev_logits": {c["id"]: float(v) for c, v in zip(data["dev"], dev_logits)},
            "test_logits": {c["id"]: float(v) for c, v in zip(data["test"], test_logits)}}
    write_outputs(out_dir, info, data["test"], (test_logits > thr).astype(int), log)
    return info


@torch.no_grad()
def student_logits(net, cases, pad_id, device, bs):
    net.eval()
    out = []
    for i in range(0, len(cases), bs):
        out += net(cases[i:i + bs], pad_id, device)[0].float().cpu().tolist()
    return np.array(out)


# ---------------------------------------------------------------------------
# One standard experiment, outputs
# ---------------------------------------------------------------------------
def run_standard(exp, data, make_backbone, device, log, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    net, info = train_stage1(exp, data, make_backbone, device, log)
    by_id = {c["id"]: c for c in data["train"]}
    agg_cases = [by_id[i] for i in data["split"]["agg_train"] + data["split"]["agg_val"]]
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t0 = time.time()
    enc = encode_cases(net, data["dev"], data["pad_id"], device, exp["infer_bs"])
    dev_s = time.time() - t0
    info["infer_cases_per_s"] = round(len(data["dev"]) / max(1e-6, dev_s), 2)
    info["peak_mem_infer_gb"] = round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0
    enc.update(encode_cases(net, data["test"], data["pad_id"], device, exp["infer_bs"]))
    enc.update(encode_cases(net, agg_cases, data["pad_id"], device, exp["infer_bs"]))
    kd_enc = None
    if exp.get("kd_teacher"):
        adapt = [by_id[i] for i in data["split"]["adapt"]]
        kd_enc = encode_cases(net, adapt, data["pad_id"], device, exp["infer_bs"], last_k=KD_LAST)
    info["encode_minutes"] = round((time.time() - t0) / 60, 1)
    log(f"[{exp['name']}] encoded chunks in {info['encode_minutes']} min ({info['infer_cases_per_s']} dev cases/s, all chunks)")
    del net
    if device.type == "cuda":
        torch.cuda.empty_cache()
    res, test_pred = stage3(exp, data, enc, device, log, out_dir, kd_enc)
    info.update(res)
    info["exp"] = exp
    write_outputs(out_dir, info, data["test"], test_pred, log)
    return info


def write_outputs(out_dir, info, test_cases, test_pred, log):
    out_dir = Path(out_dir)
    rows = []
    seen = set()
    for c, p in zip(test_cases, test_pred):
        assert c["id"] not in seen, f"duplicate test id {c['id']}"
        seen.add(c["id"])
        rows.append({"id": c["raw_id"], "pred": int(p)})
    assert len(rows) == len(test_cases) and all(r["pred"] in (0, 1) for r in rows)
    with open(out_dir / "prediction.json", "w") as f:
        json.dump(rows, f)
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    with open(out_dir / "probs.json", "w") as f:
        json.dump({"dev": info["dev_logits"], "test": info["test_logits"], "threshold": info["threshold"]}, f)
    slim = {k: v for k, v in info.items() if k not in ("dev_logits", "test_logits")}
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(slim, f, indent=1, default=str)
    r = info["dev_report"]
    log(f"[{info['exp']['name']}] DONE dev macro-F1 {r['macro_f1']:.4f} (P {r['macro_precision']:.4f} R {r['macro_recall']:.4f}) "
        f"| dev AUC {info['dev_auc']:.4f} | test predicted 1: {100 * float(np.mean(test_pred)):.1f}% -> {out_dir / 'submission.zip'}")


# ---------------------------------------------------------------------------
# Self-tests (tiny models, CPU, seconds)
# ---------------------------------------------------------------------------
def self_test(make_tiny, device, kinds=("D", "C", "A", "B")):
    torch.manual_seed(0)
    a, b = np.array([5, 9, 7, 11, 6, 8, 3, 2]), np.array([4, 10, 12])
    for kind in kinds:
        for method in ("lorma", "adalora", "unfreeze", "full") + (("bitfit",) if kind == "C" else ()):
            m, d = make_tiny(kind)
            enc = ChunkEncoder(kind, m, d).to(device).eval()
            ids, mask = pad_tokens([a, b], 0, device)
            with torch.no_grad():
                before = enc(ids, mask)
                one = enc(*pad_tokens([b], 0, device))
            assert (one[0] - before[1]).abs().max() < 1e-4, f"{kind}: padding changes the chunk vector"
            apply_method(m, method, r=4, alpha=8)
            enc = enc.to(device).eval()
            with torch.no_grad():
                after = enc(ids, mask)
            assert (before - after).abs().max() < 1e-4, f"{kind}/{method} does not start as the Phase-1 model"
            n_tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
            if method == "bitfit":
                assert n_tr > 0 and all(n.endswith(".bias") for n, p in m.named_parameters() if p.requires_grad)
            if method == "unfreeze":
                assert n_tr == 0
        # gradual unfreezing opens blocks from the top
        m, d = make_tiny(kind)
        apply_method(m, "unfreeze")
        blocks = list(blocks_of(kind, m))
        u = Unfreezer(blocks, final_norm(kind, m).parameters(), ((0, 1), (0.5, 2)), log=lambda *x: None)
        u.update(0.1)
        assert all(p.requires_grad for p in blocks[-1].parameters()) and not any(p.requires_grad for p in blocks[0].parameters())
        u.update(0.6)
        assert all(p.requires_grad for p in blocks[-2].parameters())
    # AdaLoRA: budget is reached and the forward still works
    k0 = kinds[-1]
    m, d = make_tiny(k0)
    apply_method(m, "adalora", r=4, alpha=8)
    net = ChunkModel(ChunkEncoder(k0, m, d), d).to(device)
    ada = AdaLoRA(net, 4, 2, t_init=0.1, t_final=0.3, every=1)
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=1e-2)
    for step in range(1, 21):
        ids, mask = pad_tokens([a, b], 0, device)
        loss = net(ids, mask)[0].pow(2).mean() + ada.orth_loss()
        opt.zero_grad()
        loss.backward()
        ada.update_importance()
        opt.step()
        ada.mask(step / 20, step)
    assert sum(ada.ranks().values()) <= ada.bT, ada.ranks()
    # aggregators: every kind runs; padding chunks do not change a case's logit
    X, P = torch.randn(2, 5, 16), torch.rand(2, 5, 4)
    M = torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 1]], dtype=torch.bool)
    for kind in Aggregator.KINDS:
        agg = Aggregator(16, kind, h=8).eval()
        lg, _ = agg(X, P, M)
        lg1, _ = agg(X[:1, :3], P[:1, :3], M[:1, :3])
        assert lg.shape == (2,) and abs(float((lg[0] - lg1[0]).detach())) < 1e-4, f"aggregator {kind} depends on padding"
    # student forward/backward
    ks = "C" if "C" in kinds else kinds[0]
    m, d = make_tiny(ks)
    apply_method(m, "full")
    st = HierStudent(ChunkEncoder(ks, m, d), d).to(device)
    fake = [{"toks": np.arange(1, 40, dtype=np.uint16) % 30 + 1, "chunks": [(0, 20, 0, .5), (10, 30, .25, .75), (20, 39, .5, 1)]},
            {"toks": np.arange(1, 9, dtype=np.uint16), "chunks": [(0, 8, 0, 1)]}]
    lg, w = st(fake, 0, device)
    lg.sum().backward()
    assert lg.shape == (2,) and w.shape == (2, 3)
    # calibration and metrics
    y = np.array([0, 0, 1, 1, 1, 0, 1, 0])
    s = np.array([-2, -1, 2, 1, 0.5, 0.2, -0.1, -3])
    assert abs(roc_auc(y, s) - 0.9375) < 1e-9
    thr, cal = calibrate(s, y)
    assert cal["dev_f1_threshold0"] == macro_f1(y, s > 0)
    print("cj_lib self-test passed: methods start as the Phase-1 model (LoRMA, AdaLoRA, BitFit, unfreezing, full) for "
          f"{', '.join(kinds)}; padding-safe chunk vectors; AdaLoRA reaches its rank budget; gradual unfreezing opens from the "
          "top; all 7 case readers are padding-safe; student trains; AUC / calibration ok.")


In [ ]:
%%writefile /kaggle/working/cj_train.py
"""cj_train.py -- one Phase-4 experiment: python cj_train.py <config.json> <experiment> <gpu index or -1>"""
import json
import sys
from pathlib import Path

import torch

cfg = json.load(open(sys.argv[1]))
name, gpu = sys.argv[2], int(sys.argv[3])
sys.path.insert(0, cfg["work_dir"])
import cj_lib as CJ  # noqa: E402

device = torch.device(f"cuda:{gpu}") if gpu >= 0 and torch.cuda.is_available() else torch.device("cpu")
if device.type == "cuda":
    torch.cuda.set_device(device)
exp = dict(cfg["common"], **cfg["experiments"][name], name=name)
data = torch.load(cfg["data"], weights_only=False)


def make_backbone(kind):
    """Phase-1 model from its checkpoint -> (module, d_model, kind used by the encoder)."""
    if kind == "D":
        import legal_d
        ck = torch.load(cfg["ckpt"]["D"], map_location="cpu", weights_only=False)
        m = legal_d.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        m.dec_blocks = torch.nn.ModuleList()          # encoder only: the decoder is not used for classification
        m.grad_ckpt = exp.get("grad_ckpt", True)
        return m, ck["config"]["d_model"], "D"
    if kind in ("C", "Cs"):
        import c_model as mod
    elif kind == "A":
        import a_model as mod
    else:
        import b_model as mod
    ck = torch.load(cfg["ckpt"]["C" if kind == "Cs" else kind], map_location="cpu", weights_only=False)
    m = mod.build_model(ck["config"])
    m.load_state_dict(ck["model_state"])
    mod.USE_GRAD_CHECKPOINTING = exp.get("grad_ckpt", True)
    if kind == "Cs":                                  # distillation student: the first layers of C
        m.blocks = m.blocks[:exp["student_layers"]]
    return m, ck["config"]["d_model"], "C" if kind == "Cs" else kind


def log(*a):
    print(*a, flush=True)


out_dir = Path(cfg["out_dir"]) / name
if exp.get("type") == "student":
    files = [Path(cfg["out_dir"]) / t / "kd_targets.pt" for t in exp.get("teachers", [])]
    for f in files:
        assert f.exists(), f"missing teacher targets {f}"
    CJ.run_student(exp, data, make_backbone, device, log, out_dir, files)
else:
    CJ.run_standard(exp, data, make_backbone, device, log, out_dir)


## 3. Data: download, clean, split, tokenize (T2)

In [ ]:
sys.path.insert(0, str(WORK_DIR))
import cj_lib as CJ  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

HF_TOKEN = UserSecretsClient().get_secret("HF")
REPO = "Exploration-Lab/CS-781-Capstone"
import legal_d  # noqa: E402

legal_d.load_tokenizer(TOK_PATH)
sp = legal_d.sp
PAD_ID = sp.pad_id() if sp.pad_id() >= 0 else 0
FALLBACK_ID = legal_d.EOS_ID


def encode_batch(texts):
    try:
        return sp.encode(texts, out_type=int, num_threads=os.cpu_count() or 4)
    except TypeError:
        return sp.encode(texts, out_type=int)


def hf_file(*names):
    """First of the candidate paths that exists in the repo (the capstone doc names the first one)."""
    last = None
    for n in names:
        try:
            return hf_hub_download(REPO, n, repo_type="dataset", token=HF_TOKEN)
        except Exception as e:      # noqa: BLE001
            last = e
    raise FileNotFoundError(f"none of {names} in {REPO}: {last}")


t0 = time.time()
RAW = {"train": hf_file("Phase_4_CJPE/Train/train.json", "Phase_4_CJPE/Train/train.jsonl"),
       "dev": hf_file("Phase_4_CJPE/Train/dev.json", "Phase_4_CJPE/Train/dev.jsonl"),
       "test": hf_file("Phase_4_CJPE/Test/test.json", "Phase_4_CJPE/Test/test.jsonl")}
cases = {s: CJ.parse_cases(CJ.load_json_any(p)) for s, p in RAW.items()}
print(f"downloaded + parsed in {time.time() - t0:.0f}s:", {s: len(v) for s, v in cases.items()})
for s in ("train", "dev"):
    assert all(c["label"] in (0, 1) for c in cases[s]), f"{s}: every case needs a 0/1 label"
test_ids = [c["id"] for c in cases["test"]]
assert len(test_ids) == len(set(test_ids)), "test: duplicate case ids"
# dev repeats some case ids: drop exact repeats (same id, text, label), rename the rest id#2, id#3, ...
seen_dev, n_used, dev_clean, n_rep = set(), {}, [], 0
for c in cases["dev"]:
    key = (c["id"], hashlib.md5(c["text"].encode("utf-8")).hexdigest(), c["label"])
    if key in seen_dev:
        n_rep += 1
        continue
    seen_dev.add(key)
    n = n_used.get(c["id"], 0)
    n_used[c["id"]] = n + 1
    dev_clean.append(dict(c, id=f"{c['id']}#{n + 1}") if n else c)
print(f"dev: {len(cases['dev'])} rows, {n_rep} exact repeats dropped, "
      f"{sum(1 for c in dev_clean if '#' in c['id'])} renamed -> {len(dev_clean)} dev cases")
cases["dev"] = dev_clean
print("example:", {k: (v[:120] + "...") if isinstance(v, str) else v for k, v in cases["train"][0].items()})

# exact duplicates: drop train cases whose text is also in dev/test, keep one copy of repeated train
# cases, drop texts that appear with both labels; make internal train ids unique
h = lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()
held = {h(c["text"]) for s in ("dev", "test") for c in cases[s]}
first, labels_of, n_overlap, n_dup = {}, {}, 0, 0
for c in cases["train"]:
    k = h(c["text"])
    if k in held:
        n_overlap += 1
        continue
    labels_of.setdefault(k, set()).add(c["label"])
    if k in first:
        n_dup += 1
        continue
    first[k] = c
conflict = {k for k, v in labels_of.items() if len(v) > 1}
train, seen_ids = [], {}
for k, c in first.items():
    if k in conflict:
        continue
    n = seen_ids.get(c["id"], 0)
    seen_ids[c["id"]] = n + 1
    if n:
        c = dict(c, id=f"{c['id']}#{n + 1}")
    train.append(c)
cases["train"] = train
print(f"train cleaning: {n_overlap} also in dev/test (dropped), {n_dup} duplicate copies (dropped), "
      f"{len(conflict)} texts with both labels (dropped) -> {len(train)} train cases")
for s in ("train", "dev"):
    y = [c["label"] for c in cases[s]]
    print(f"{s}: {len(y)} cases, {100 * np.mean(y):.1f}% label 1")
words = np.array([len(c["text"].split()) for c in cases["train"]])
print(f"train case length (words): median {np.median(words):.0f}, 90th pct {np.quantile(words, 0.9):.0f}, max {words.max()}")

t0 = time.time()
data = {"pad_id": PAD_ID, "tokenizer": "T2"}
for s in ("train", "dev", "test"):
    data[s] = CJ.tokenize_cases(cases[s], encode_batch, FALLBACK_ID)
data["split"] = CJ.split_train(data["train"])
ntok = np.array([c["n_tok"] for c in data["train"]])
nch = np.array([len(c["chunks"]) for c in data["train"]])
print(f"tokenized in {time.time() - t0:.0f}s | train tokens per case: median {np.median(ntok):.0f}, 90th pct "
      f"{np.quantile(ntok, 0.9):.0f} | chunks per case: mean {nch.mean():.1f}, max {nch.max()} | "
      f"{100 * np.mean([c['cut'] for c in data['train']]):.1f}% of cases longer than {CJ.MAX_TOKENS} tokens (middle skipped)")
print("split:", {k: len(v) for k, v in data["split"].items()})
DATA_PATH = OUT_DIR / "cj_data.pt"
torch.save(data, DATA_PATH)
DEV_LABELS = {c["id"]: c["label"] for c in data["dev"]}
json.dump(DEV_LABELS, open(OUT_DIR / "dev_labels.json", "w"))
del cases, train, first, data
gc.collect()


## 4. Self-tests (tiny models, a few seconds)

In [ ]:
import c_model  # noqa: E402


def make_tiny(kind):
    if kind == "D":
        cfg = dict(legal_d.MODEL_D_CONFIG, vocab_size=64, d_model=32, n_enc_layer=2, n_dec_layer=1, n_head=4,
                   n_kv_head=2, d_head=8, d_ff=48, max_enc_len=64, max_dec_len=64)
        m = legal_d.build_model(cfg).eval()
        m.dec_blocks = torch.nn.ModuleList()
        return m, 32
    if kind == "C":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=64,
                   qk_norm=True, bias=True, mlp="gelu", tokenizer_id="T2", attn_type="prefix", tied_embeddings=True)
        return c_model.build_model(cfg).eval(), 32
    raise KeyError(kind)


CJ.self_test(make_tiny, torch.device("cpu"), kinds=("D", "C"))


## 5. Experiment settings

In [ ]:
COMMON = dict(epochs=2, seed=13, lr_head=1e-3, grad_ckpt=True)
EXPERIMENTS = {
    "S1": dict(title="D encoder + LoRMA", model="D", method="lorma", r=16, alpha=32, lr=5e-4,
               micro_bs=16, accum=2, budget_min=60, infer_bs=64, kd_teacher=True, submit=True),
    "S2": dict(title="C + BitFit", model="C", method="bitfit", lr=3e-3,
               micro_bs=16, accum=2, budget_min=60, infer_bs=48, kd_teacher=True, submit=True),
    "R1": dict(title="D encoder full fine-tune (reference)", model="D", method="full", lr=2e-5,
               micro_bs=16, accum=2, budget_min=60, infer_bs=64, submit=False),
    "S5": dict(title="KD student: C first 6 layers <- S1 + S2", type="student", model="Cs", student_layers=6,
               lr=5e-5, micro_bs=4, accum=4, budget_min=75, infer_bs=16, teachers=["S1", "S2"],
               teacher_names="S1 (D + LoRMA) + S2 (C + BitFit)", kd_alpha=0.5, kd_T=2.0, attn_beta=1.0, submit=True),
    "R2": dict(title="Student, gold labels only (control)", type="student", model="Cs", student_layers=6,
               lr=5e-5, micro_bs=4, accum=4, budget_min=75, infer_bs=16, teachers=[], kd_alpha=0.0, attn_beta=0.0,
               submit=False),
}
RUN_ORDER = [("S1", []), ("S2", []), ("R1", []), ("S5", ["S1", "S2"]), ("R2", [])]   # (run, must finish first)
CFG = {"work_dir": str(WORK_DIR), "data": str(DATA_PATH), "ckpt": CKPT, "out_dir": str(OUT_DIR),
       "common": COMMON, "experiments": EXPERIMENTS}
CONFIG_PATH = WORK_DIR / "cj_config.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=1)
for name, e in EXPERIMENTS.items():
    print(f"{name}: {e['title']}")


## 6. Run (two at a time, one per GPU)

In [ ]:


def stream(proc, tag):
    for line in proc.stdout:
        print(line if line.startswith(f"[{tag}]") else f"[{tag}] {line}", end="", flush=True)


queue, running, codes = list(RUN_ORDER), {}, {}
slots = list(range(N_GPUS))[:2] if N_GPUS else [-1]
t_all = time.time()
while queue or running:
    for slot in slots:
        if slot in running:
            continue
        ready = [q for q in queue if all(d in codes for d in q[1])]
        if not ready:
            continue
        name, deps = ready[0]
        queue.remove(ready[0])
        if any(codes.get(d) != 0 for d in deps):
            print(f">>> skipping {name}: {deps} did not all finish", flush=True)
            codes[name] = -1
            continue
        env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        proc = subprocess.Popen([sys.executable, str(WORK_DIR / "cj_train.py"), str(CONFIG_PATH), name, str(slot)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        th = threading.Thread(target=stream, args=(proc, name), daemon=True)
        th.start()
        running[slot] = (name, proc, th)
        print(f">>> started {name} ({EXPERIMENTS[name]['title']}) on {'GPU ' + str(slot) if slot >= 0 else 'CPU'}", flush=True)
    for slot, (name, proc, th) in list(running.items()):
        if proc.poll() is not None:
            th.join()
            codes[name] = proc.returncode
            print(f">>> {name} finished with code {proc.returncode} ({(time.time() - t_all) / 60:.1f} min so far)", flush=True)
            del running[slot]
    time.sleep(5)
print("Exit codes:", codes)


## 7. Results, analysis tables, submissions

In [ ]:
rows, by = [], {}
for name, _ in RUN_ORDER:
    mpath = OUT_DIR / name / "metrics.json"
    if not mpath.exists():
        print(f"{name}: no metrics (see its log above)")
        continue
    m = json.load(open(mpath))
    e = EXPERIMENTS[name]
    r = {"run": name, "title": e["title"], "model": e["model"], "method": e.get("method", "distillation" if e.get("teachers") else "student"),
         "backbone_M": round(m["n_backbone"] / 1e6, 1), "backbone_params": m["n_backbone"], "trainable": m["n_trainable"], "trainable_pct": m["trainable_pct"],
         "dev_macro_f1": m["dev_report"]["macro_f1"], "dev_precision": m["dev_report"]["macro_precision"],
         "dev_recall": m["dev_report"]["macro_recall"], "dev_auc": m["dev_auc"], "reader": m["chosen_reader"],
         "threshold": m["threshold"], "f1_threshold0": m["calibration"]["dev_f1_threshold0"],
         "train_min": m.get("stage1_minutes", m.get("train_minutes")), "encode_min": m.get("encode_minutes", 0),
         "peak_mem_train_gb": m["peak_mem_train_gb"], "peak_mem_infer_gb": m["peak_mem_infer_gb"],
         "infer_cases_per_s": m["infer_cases_per_s"], "submit": e["submit"]}
    rows.append(r)
    by[name] = r
assert rows, "No run finished -- read the error lines above."
with open(OUT_DIR / "runs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print(f"{'run':4s} {'title':42s} {'trainable':>12s} {'%':>7s} {'F1':>7s} {'P':>7s} {'R':>7s} {'AUC':>7s} {'min':>6s} {'GB':>5s} {'case/s':>7s}")
for r in rows:
    print(f"{r['run']:4s} {r['title'][:42]:42s} {r['trainable']:>12,} {r['trainable_pct']:7.2f} {r['dev_macro_f1']:7.4f} "
          f"{r['dev_precision']:7.4f} {r['dev_recall']:7.4f} {r['dev_auc']:7.4f} {r['train_min'] + r['encode_min']:6.1f} "
          f"{r['peak_mem_train_gb']:5.1f} {r['infer_cases_per_s']:7.2f}")

# long-document readers and the section study, per run (dev numbers are for reporting; choice used val AUC)
for name in by:
    m = json.load(open(OUT_DIR / name / "metrics.json"))
    if "readers" not in m:
        continue
    print(f"\n{name} -- case readers (chosen: {m['chosen_reader']}):")
    for x in m["readers"]:
        print(f"  {x['reader']:30s} val AUC {x['val_auc']:.4f} | dev AUC {x['dev_auc']:.4f} | dev macro-F1 {x['dev_macro_f1_at_0']:.4f}  ({x['note']})")
    print("  attention share by case position (deciles, start -> end):", m["attention_by_position_dev"])
    print("  errors by case length:", m["errors_by_length"])

# distillation: teachers vs student vs the same student without distillation
dl = json.load(open(OUT_DIR / "dev_labels.json"))
probs = {n: json.load(open(OUT_DIR / n / "probs.json")) for n in ("S1", "S2") if (OUT_DIR / n / "probs.json").exists()}
if len(probs) == 2:
    ids = list(dl)
    ens = np.array([np.mean([probs[n]["dev"][i] for n in probs]) for i in ids])
    y = [dl[i] for i in ids]
    print(f"\nTeacher ensemble S1 + S2 (mean logit, threshold 0): dev macro-F1 {CJ.macro_f1(y, ens > 0):.4f} | AUC {CJ.roc_auc(y, ens):.4f}")
    t_params = sum(by[n]["backbone_params"] for n in ("S1", "S2") if n in by)
    t_speed = 1 / sum(1 / max(1e-9, by[n]["infer_cases_per_s"]) for n in ("S1", "S2") if n in by)
    print(f"  teachers: {t_params / 1e6:.1f}M parameters, {t_speed:.2f} dev cases/s (both read every chunk)")
    for n in ("S5", "R2"):
        if n in by:
            r = by[n]
            print(f"  {n} {r['title']}: {r['backbone_params'] / 1e6:.1f}M ({100 * r['backbone_params'] / max(1, t_params):.0f}% of the teachers), "
                  f"{r['infer_cases_per_s']:.2f} cases/s (last 4 chunks), dev macro-F1 {r['dev_macro_f1']:.4f}, AUC {r['dev_auc']:.4f}")

ens = {}
for name in by:
    p = json.load(open(OUT_DIR / name / "probs.json"))
    ens[name] = dict(p, title=EXPERIMENTS[name]["title"], dev_macro_f1=by[name]["dev_macro_f1"])
ens["dev_labels"] = json.load(open(OUT_DIR / "dev_labels.json"))
json.dump(ens, open(OUT_DIR / "ens_nb1.json", "w"))
print("\nSubmit these (Output tab):")
for name, e in EXPERIMENTS.items():
    if e["submit"]:
        z = OUT_DIR / name / "submission.zip"
        print(f"  {name} ({e['title']}): {z}" if z.exists() else f"  {name}: MISSING")
print(f"Also download {OUT_DIR / 'ens_nb1.json'} for the ensemble (S6).")
